# shap 값

In [1]:
# ============================================================
# 전체 업종 Top10 1위 조합 기반 SHAP + Permutation Importance
# 저장: 중간결과\16_SHAP\{prefix}\
#   - feature_analysis_{FEATURE_SET}.csv
#   - shap_bar_{FEATURE_SET}.png
#   - shap_beeswarm_{FEATURE_SET}.png
#   - permutation_bar_{FEATURE_SET}.png
# ============================================================

import os
import re
import glob
import platform
import warnings
import numpy as np
import pandas as pd
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("[경고] LightGBM 미설치 → LightGBM 모델 사용 불가")

try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

try:
    from ctgan import CTGAN
    HAS_CTGAN = True
except ImportError:
    HAS_CTGAN = False

warnings.filterwarnings("ignore")

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False


# ============================================================
# 1. 전역 경로 설정
# ============================================================
BASE_DIR       = r'중간결과'
TOP10_FOLDER   = os.path.join(BASE_DIR, r'15_우수모델\Top10')
TRAIN_FOLDER   = os.path.join(BASE_DIR, '12_train')
TEST_FOLDER    = os.path.join(BASE_DIR, '12_test')
FEATURE_FOLDER = os.path.join(BASE_DIR, '13_피처셀렉션')
SHAP_SAVE_DIR  = os.path.join(BASE_DIR, '16_SHAP')

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42


# ============================================================
# 2. 업종 정보 파싱 유틸
# ============================================================

def parse_industry_info(top10_csv_path: str):
    fname = os.path.basename(top10_csv_path)
    match = re.match(r'^(.+?)_Top10', fname)
    if not match:
        raise ValueError(f"파일명 패턴 불일치: {fname}")
    prefix = match.group(1)   # e.g. M03_음식료품_제조업
    return {
        "prefix"         : prefix,
        "train_file"     : os.path.join(TRAIN_FOLDER, f"{prefix}_train.parquet"),
        "test_file"      : os.path.join(TEST_FOLDER,  f"{prefix}_test.parquet"),
        "feature_folder" : os.path.join(FEATURE_FOLDER, prefix),
    }


# ============================================================
# 3. 모델 팩토리
# ============================================================

def make_model(model_name: str, pos_weight: float):
    mn = str(model_name).strip()

    if mn == "XGBoost":
        return XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="aucpr",
            random_state=RANDOM_STATE, verbosity=0,
            scale_pos_weight=pos_weight
        )

    if mn == "LightGBM":
        if not HAS_LGBM:
            raise ImportError("LightGBM 미설치. pip install lightgbm")
        if pos_weight > 1.0:
            return LGBMClassifier(
                n_estimators=300, learning_rate=0.05, max_depth=4,
                subsample=0.8, colsample_bytree=0.8,
                random_state=RANDOM_STATE, verbose=-1,
                scale_pos_weight=pos_weight
            )
        return LGBMClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8,
            random_state=RANDOM_STATE, verbose=-1
        )

    if mn == "RandomForest":
        if pos_weight > 1.0:
            return RandomForestClassifier(
                n_estimators=300, max_depth=4, min_samples_leaf=5,
                random_state=RANDOM_STATE, n_jobs=-1,
                class_weight="balanced"
            )
        return RandomForestClassifier(
            n_estimators=300, max_depth=4, min_samples_leaf=5,
            random_state=RANDOM_STATE, n_jobs=-1
        )

    if mn == "LogisticRegression":
        if pos_weight > 1.0:
            return LogisticRegression(
                penalty="l2", C=1.0, solver="lbfgs",
                max_iter=1000, random_state=RANDOM_STATE,
                class_weight="balanced"
            )
        return LogisticRegression(
            penalty="l2", C=1.0, solver="lbfgs",
            max_iter=1000, random_state=RANDOM_STATE
        )

    raise ValueError(f"지원하지 않는 모델명: {model_name}")


# ============================================================
# 4. 불균형 처리 함수
# ============================================================

def _parse_smote_ratio(smote_ratio):
    try:
        if smote_ratio is None:
            return None
        s = str(smote_ratio).strip()
        if s in ("", "-", "nan", "None"):
            return None
        return float(s)
    except (TypeError, ValueError):
        return None


def _ctgan_resample(X, y, ratio):
    if not HAS_CTGAN:
        raise ImportError("CTGAN 방식을 사용하려면 'pip install ctgan'이 필요합니다.")
    if not isinstance(X, pd.DataFrame):
        raise TypeError("_ctgan_resample: X는 pd.DataFrame이어야 합니다.")

    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)

    n_minority = int((y == 1).sum())
    n_majority = int((y == 0).sum())

    if n_minority < 5:
        print(f"    [CTGAN 경고] 소수 클래스({n_minority}개) 너무 적어 합성 건너뜁니다.")
        return X, y

    target_minority = int(n_majority * ratio) if ratio is not None else n_majority
    n_synth = target_minority - n_minority

    if n_synth <= 0:
        print(f"    [CTGAN] 현재 소수({n_minority}) >= 목표({target_minority}), 합성 불필요.")
        return X, y

    print(f"    [CTGAN] 소수={n_minority}, 다수={n_majority}, ratio={ratio}, 합성수={n_synth}")
    minority_df = X[y == 1].copy().reset_index(drop=True)

    try:
        ctgan = CTGAN(epochs=300, verbose=False)
    except TypeError:
        ctgan = CTGAN(epochs=300)

    ctgan.fit(minority_df)
    synth = ctgan.sample(n_synth).reindex(columns=X.columns)

    X_res = pd.concat([X, synth], ignore_index=True)
    y_res = pd.concat([y, pd.Series([1] * n_synth)], ignore_index=True)
    print(f"    [CTGAN] 완료 → 정상={int((y_res==0).sum())}, 부실={int((y_res==1).sum())}")
    return X_res, y_res


def apply_resampling(X, y, method, smote_ratio):
    method_l = str(method).strip().lower()
    ratio    = _parse_smote_ratio(smote_ratio)

    if method_l == "classweight":
        pos_weight = (y == 0).sum() / (y == 1).sum()
        return X, y, pos_weight

    if method_l in ("none", "", "-", "nan"):
        return X, y, 1.0

    if "borderline" in method_l:
        if not HAS_IMBLEARN:
            raise ImportError("pip install imbalanced-learn")
        sampler = BorderlineSMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if method_l == "smote":
        if not HAS_IMBLEARN:
            raise ImportError("pip install imbalanced-learn")
        sampler = SMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if "ctgan" in method_l:
        X_res, y_res = _ctgan_resample(X, y, ratio)
        return X_res, y_res, 1.0

    print(f"  [경고] 알 수 없는 Method='{method}' → ClassWeight로 처리합니다.")
    pos_weight = (y == 0).sum() / (y == 1).sum()
    return X, y, pos_weight


# ============================================================
# 5. SHAP Explainer 선택 (모델 종류별)
# ============================================================

def get_shap_values(model, model_name: str, X: pd.DataFrame):
    """
    - Tree 계열 (XGBoost, LightGBM, RandomForest): TreeExplainer
    - 그 외 (LogisticRegression 등): LinearExplainer (또는 KernelExplainer fallback)
    반환: shap_values (2D array, shape=[n_samples, n_features])
    """
    mn = str(model_name).strip()

    if mn in ("XGBoost", "LightGBM", "RandomForest"):
        explainer   = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X)

        # RandomForest는 클래스별 list 반환 → 양성 클래스(index 1) 선택
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        return shap_values

    # LogisticRegression → LinearExplainer
    if mn == "LogisticRegression":
        explainer   = shap.LinearExplainer(model, X)
        shap_values = explainer.shap_values(X)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        return shap_values

    # fallback: KernelExplainer (느리므로 샘플 200개 제한)
    print(f"  [SHAP] 알 수 없는 모델({mn}) → KernelExplainer 사용 (샘플 200개)")
    bg = shap.sample(X, 100)
    explainer   = shap.KernelExplainer(model.predict_proba, bg)
    shap_values = explainer.shap_values(X.sample(min(200, len(X)),
                                                   random_state=RANDOM_STATE))
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    return shap_values


# ============================================================
# 6. 피처 카테고리 매핑
#    - 베이스 피처명(접미사 제거 전)으로 카테고리 정의
#    - _diff / _ratio / _diff_industry / _ratio_industry 등
#      모든 변형 컬럼은 get_category() 함수에서 접미사 제거 후 조회
# ============================================================

# 베이스 피처명 → 카테고리 (접미사 없는 원형 기준)
FEATURE_CATEGORY_BASE = {
    # ── 안정성 (Solvency) ──────────────────────────────────
    "부채비율"              : "안정성 (Solvency)",
    "총부채비율"            : "안정성 (Solvency)",
    "장기부채비율"          : "안정성 (Solvency)",
    "장기부채의존도"        : "안정성 (Solvency)",
    "차입금의존도"          : "안정성 (Solvency)",
    "순차입금비율"          : "안정성 (Solvency)",
    "금융부채비율"          : "안정성 (Solvency)",
    "자기자본비율"          : "안정성 (Solvency)",
    "유보율"                : "안정성 (Solvency)",
    "자본잠식률"            : "안정성 (Solvency)",
    "유동비율"              : "안정성 (Solvency)",
    "당좌비율"              : "안정성 (Solvency)",
    "당좌비율_추정"         : "안정성 (Solvency)",
    "현금비율"              : "안정성 (Solvency)",
    "순운전자본비율"        : "안정성 (Solvency)",
    "순운전자본대총자본"    : "안정성 (Solvency)",
    "비유동비율"            : "안정성 (Solvency)",
    "비유동장기적합률"      : "안정성 (Solvency)",
    "유형자산부채비율"      : "안정성 (Solvency)",
    "부채비율변화"          : "안정성 (Solvency)",
    "유동비율변화"          : "안정성 (Solvency)",

    # ── 수익성 (Profitability) ─────────────────────────────
    "ROA"                   : "수익성 (Profitability)",
    "ROE"                   : "수익성 (Profitability)",
    "ROIC"                  : "수익성 (Profitability)",
    "총자본영업이익률"      : "수익성 (Profitability)",
    "매출총이익률"          : "수익성 (Profitability)",
    "영업이익률"            : "수익성 (Profitability)",
    "순이익률"              : "수익성 (Profitability)",
    "EBITDA마진"            : "수익성 (Profitability)",
    "현금ROA"               : "수익성 (Profitability)",
    "현금ROE"               : "수익성 (Profitability)",
    "매출원가율"            : "수익성 (Profitability)",
    "판관비율"              : "수익성 (Profitability)",
    "금융비용부담률"        : "수익성 (Profitability)",
    "ROA변화"               : "수익성 (Profitability)",
    "영업이익률변화"        : "수익성 (Profitability)",
    "매출액순이익률"        : "수익성 (Profitability)",
    "매출액영업이익률"      : "수익성 (Profitability)",
    "EBIT대매출액"          : "수익성 (Profitability)",
    "EBITDA대매출액"        : "수익성 (Profitability)",
    "매출원가대매출액"      : "수익성 (Profitability)",
    "금융비용대매출액"      : "수익성 (Profitability)",
    "사내유보율"            : "수익성 (Profitability)",

    # ── 성장성 (Growth) ────────────────────────────────────
    "매출액증가율"          : "성장성 (Growth)",
    "영업이익증가율"        : "성장성 (Growth)",
    "순이익증가율"          : "성장성 (Growth)",
    "EBITDA증가율"          : "성장성 (Growth)",
    "총자산증가율"          : "성장성 (Growth)",
    "유형자산증가율"        : "성장성 (Growth)",
    "자기자본증가율"        : "성장성 (Growth)",
    "부채증가율"            : "성장성 (Growth)",
    "영업현금흐름증가율"    : "성장성 (Growth)",
    "FCF증가율"             : "성장성 (Growth)",

    # ── 활동성 (Activity) ──────────────────────────────────
    "총자산회전율"          : "활동성 (Activity)",
    "유동자산회전율"        : "활동성 (Activity)",
    "비유동자산회전율"      : "활동성 (Activity)",
    "유형자산회전율"        : "활동성 (Activity)",
    "자기자본회전율"        : "활동성 (Activity)",
    "투하자본회전율"        : "활동성 (Activity)",
    "매출채권회전율"        : "활동성 (Activity)",
    "매출채권회수기간"      : "활동성 (Activity)",
    "재고자산회전율"        : "활동성 (Activity)",
    "재고자산보유기간"      : "활동성 (Activity)",
    "매입채무회전율"        : "활동성 (Activity)",
    "매입채무지급기간"      : "활동성 (Activity)",
    "현금전환주기_CCC"      : "활동성 (Activity)",
    "순운전자본회전율"      : "활동성 (Activity)",

    # ── 현금흐름 (Cash Flow) ───────────────────────────────
    "영업현금흐름비율"      : "현금흐름 (Cash Flow)",
    "영업CF_유동부채"       : "현금흐름 (Cash Flow)",
    "영업CF_총부채"         : "현금흐름 (Cash Flow)",
    "FCF_총자산"            : "현금흐름 (Cash Flow)",
    "감가상각비율"          : "현금흐름 (Cash Flow)",

    # ── 기타 (Other) ───────────────────────────────────────
    "업력"                  : "기타 (Other)",
    "빅4감사"               : "기타 (Other)",
    "유형자산비율"          : "기타 (Other)",
    "종업원"                : "기타 (Other)",
}

# 접미사 제거 순서 (긴 것부터 먼저 제거해야 오매칭 방지)
_SUFFIXES = [
    "_diff_industry",
    "_ratio_industry",
    "_diff",
    "_ratio",
]

def get_category(feature_name: str) -> str:
    """
    피처명에서 접미사(_diff/_ratio/_diff_industry/_ratio_industry)를
    순서대로 제거한 뒤 FEATURE_CATEGORY_BASE에서 카테고리를 찾는다.
    직접 매핑이 없으면 접미사 제거 후 재시도.
    """
    # 1) 완전 일치 우선
    if feature_name in FEATURE_CATEGORY_BASE:
        return FEATURE_CATEGORY_BASE[feature_name]

    # 2) 접미사 제거 후 재시도
    base = feature_name
    for sfx in _SUFFIXES:
        if base.endswith(sfx):
            base = base[: -len(sfx)]
            break   # 한 번만 제거

    if base in FEATURE_CATEGORY_BASE:
        return FEATURE_CATEGORY_BASE[base]

    return "미분류"


CATEGORY_ORDER = {
    "안정성 (Solvency)"      : 1,
    "수익성 (Profitability)" : 2,
    "성장성 (Growth)"        : 3,
    "활동성 (Activity)"      : 4,
    "현금흐름 (Cash Flow)"   : 5,
    "기타 (Other)"           : 6,
    "미분류"                 : 7,
}


# ============================================================
# 7. 단일 업종 처리 함수
# ============================================================

def process_one_industry(top10_csv_path: str):
    info   = parse_industry_info(top10_csv_path)
    prefix = info["prefix"]

    print(f"\n{'='*70}")
    print(f"[업종] {prefix}")
    print(f"{'='*70}")

    # ── Top10 1위 행 로드
    top10 = pd.read_csv(top10_csv_path, index_col=0)
    if len(top10) == 0:
        print(f"  [경고] 파일이 비어있습니다. skip.")
        return None

    row          = top10.iloc[0]
    FEATURE_SET  = row["FeatureSet"]
    FEATURE_FILE = row["FeatureFile"]
    METHOD       = row["Method"]
    SMOTE_RATIO  = row["SMOTE_Ratio"]
    MODEL_NAME   = row["Model"]

    print(f"  Model={MODEL_NAME} | Method={METHOD} | "
          f"FeatureSet={FEATURE_SET} | SMOTE_Ratio={SMOTE_RATIO}")

    # ── 저장 폴더
    save_sub = os.path.join(SHAP_SAVE_DIR, prefix)
    os.makedirs(save_sub, exist_ok=True)

    # ── 파일 존재 확인
    feature_path = os.path.join(info["feature_folder"], FEATURE_FILE)
    for label, path in [("피처 파일", feature_path),
                         ("Train",    info["train_file"]),
                         ("Test",     info["test_file"])]:
        if not os.path.exists(path):
            print(f"  [경고] {label} 없음: {path} → skip")
            return None

    # ── 데이터 로드
    train_full = pd.read_parquet(info["train_file"])
    test       = pd.read_parquet(info["test_file"])
    y_train    = train_full[TARGET_COL]
    y_test     = test[TARGET_COL]

    feat_df      = pd.read_csv(feature_path)
    col_key      = "feature" if "feature" in feat_df.columns else feat_df.columns[0]
    use_features = [f for f in feat_df[col_key].tolist() if f in train_full.columns]
    print(f"  피처 수: {len(use_features)}개")

    # ── 결측치 처리
    imputer     = SimpleImputer(strategy="median")
    X_train_imp = pd.DataFrame(
        imputer.fit_transform(train_full[use_features]), columns=use_features
    )
    X_test_imp  = pd.DataFrame(
        imputer.transform(test[use_features]), columns=use_features
    )

    # ── 리샘플링 + 모델 학습
    X_res, y_res, pos_weight = apply_resampling(X_train_imp, y_train, METHOD, SMOTE_RATIO)
    print(f"  리샘플링({METHOD}): {len(X_train_imp)}행 → {len(X_res)}행 | pos_weight={pos_weight:.4f}")

    model = make_model(MODEL_NAME, pos_weight)
    model.fit(X_res, y_res)
    print(f"  모델 학습 완료 ({MODEL_NAME})")

    # ── SHAP 계산
    print(f"  SHAP 계산 중...")
    shap_values = get_shap_values(model, MODEL_NAME, X_test_imp)

    shap_importance = pd.DataFrame({
        "Feature"       : use_features,
        "mean_abs_SHAP" : np.abs(shap_values).mean(axis=0)
    }).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)
    shap_importance["Rank"] = shap_importance.index + 1

    # ── PNG 1: SHAP Bar
    plt.figure(figsize=(10, max(6, len(use_features) * 0.3)))
    shap.summary_plot(shap_values, X_test_imp, plot_type="bar", show=False, max_display=20)
    plt.title(f"SHAP Global Feature Importance (Bar)\n{prefix} | {FEATURE_SET} ({METHOD})",
              fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(save_sub, f"shap_bar_{FEATURE_SET}.png"),
                dpi=150, bbox_inches="tight")
    plt.close()

    # ── PNG 2: SHAP Beeswarm
    plt.figure(figsize=(10, max(6, len(use_features) * 0.3)))
    shap.summary_plot(shap_values, X_test_imp, plot_type="dot", show=False, max_display=20)
    plt.title(f"SHAP Beeswarm Plot\n{prefix} | {FEATURE_SET} ({METHOD})", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(save_sub, f"shap_beeswarm_{FEATURE_SET}.png"),
                dpi=150, bbox_inches="tight")
    plt.close()

    # ── Permutation Importance
    print(f"  Permutation Importance 계산 중...")
    perm_result = permutation_importance(
        model, X_test_imp, y_test,
        n_repeats=30, random_state=RANDOM_STATE,
        scoring="average_precision", n_jobs=-1
    )
    perm_df = pd.DataFrame({
        "Feature"   : use_features,
        "Perm_Mean" : perm_result.importances_mean,
        "Perm_Std"  : perm_result.importances_std,
    }).sort_values("Perm_Mean", ascending=False).reset_index(drop=True)
    perm_df["Rank"] = perm_df.index + 1

    # ── PNG 3: Permutation Bar
    top_n    = min(20, len(use_features))
    perm_top = perm_df.head(top_n).sort_values("Perm_Mean", ascending=True)
    fig, ax  = plt.subplots(figsize=(10, max(6, top_n * 0.4)))
    ax.barh(
        perm_top["Feature"], perm_top["Perm_Mean"],
        xerr=perm_top["Perm_Std"],
        color="#2F6EBA", alpha=0.8,
        error_kw=dict(ecolor="#555555", capsize=3)
    )
    ax.axvline(0, color="red", linestyle="--", linewidth=1.0)
    ax.set_xlabel("Mean decrease in PR_AUC", fontsize=10)
    ax.set_title(f"Permutation Importance (Top {top_n})\n{prefix} | {FEATURE_SET} ({METHOD})",
                 fontsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(save_sub, f"permutation_bar_{FEATURE_SET}.png"),
                dpi=150, bbox_inches="tight")
    plt.close()

    # ── feature_analysis CSV 생성
    n_features = len(shap_importance)

    df = shap_importance[["Rank", "Feature", "mean_abs_SHAP"]].rename(
        columns={"Rank": "SHAP_Rank"}
    ).merge(
        perm_df[["Feature", "Perm_Mean", "Perm_Std", "Rank"]].rename(
            columns={"Rank": "Perm_Rank"}
        ),
        on="Feature", how="inner"
    )

    df["SHAP_Rank_Norm"] = (df["SHAP_Rank"] - 1) / max(n_features - 1, 1)
    df["Perm_Rank_Norm"] = (df["Perm_Rank"] - 1) / max(n_features - 1, 1)
    df["Rank_Diff"]      = (df["SHAP_Rank"] - df["Perm_Rank"]).abs()
    df["Combined_Score"] = (df["SHAP_Rank_Norm"] + df["Perm_Rank_Norm"]) / 2
    df["Combined_Rank"]  = df["Combined_Score"].rank(method="min").astype(int)

    top_n_cls    = max(3, int(n_features * 0.3))
    shap_top_set = set(df.nsmallest(top_n_cls, "SHAP_Rank")["Feature"])
    perm_top_set = set(df.nsmallest(top_n_cls, "Perm_Rank")["Feature"])

    def classify(r):
        in_shap = r["Feature"] in shap_top_set
        in_perm = r["Feature"] in perm_top_set
        if   in_shap and in_perm:     return "★ 핵심피처 (SHAP+Perm 모두 높음)"
        elif in_shap and not in_perm: return "△ 대체가능 (SHAP 높음, Perm 낮음)"
        elif not in_shap and in_perm: return "▲ 상호작용 (Perm 높음, SHAP 낮음)"
        else:                         return "- 일반피처"

    def rank_diff_level(diff):
        if diff <= 3:   return "일치"
        elif diff <= 8: return "소폭 불일치"
        else:           return "대폭 불일치"

    df["Feature_Type"]   = df.apply(classify, axis=1)
    df["Consistency"]    = df["Rank_Diff"].apply(rank_diff_level)
    df["Category"]       = df["Feature"].apply(get_category)
    df["Category_Order"] = df["Category"].map(CATEGORY_ORDER)

    # 미분류 피처 있으면 출력 (디버깅용)
    unclassified = df[df["Category"] == "미분류"]["Feature"].tolist()
    if unclassified:
        print(f"  [카테고리 미분류 피처 {len(unclassified)}개]: {unclassified}")

    df = df[[
        "Combined_Rank", "Feature", "Category", "Feature_Type",
        "SHAP_Rank", "mean_abs_SHAP",
        "Perm_Rank", "Perm_Mean", "Perm_Std",
        "Rank_Diff", "Consistency",
        "Combined_Score", "Category_Order",
    ]].sort_values("Combined_Rank").reset_index(drop=True)

    csv_path = os.path.join(save_sub, f"feature_analysis_{FEATURE_SET}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"  저장 완료:")
    print(f"    → {csv_path}")
    print(f"    → shap_bar_{FEATURE_SET}.png")
    print(f"    → shap_beeswarm_{FEATURE_SET}.png")
    print(f"    → permutation_bar_{FEATURE_SET}.png")

    return {"Industry": prefix, "FeatureSet": FEATURE_SET, "Model": MODEL_NAME,
            "Method": METHOD, "N_Features": n_features}


# ============================================================
# 8. 전체 Top10 파일 순회 실행
# ============================================================

def main():
    top10_files = sorted(glob.glob(os.path.join(TOP10_FOLDER, "*.csv")))
    if not top10_files:
        print(f"[오류] Top10 파일이 없습니다: {TOP10_FOLDER}")
        return

    # ── 스킵할 업종 (CTGAN 등 시간 소요가 큰 업종)
    SKIP_KEYWORDS = [
    ]

    skip_files = [f for f in top10_files
                  if any(kw in os.path.basename(f) for kw in SKIP_KEYWORDS)]
    run_files  = [f for f in top10_files if f not in skip_files]

    print(f"전체 업종 수  : {len(top10_files)}개")
    print(f"스킵 업종 수  : {len(skip_files)}개")
    for f in skip_files:
        print(f"  [SKIP] {os.path.basename(f)}")
    print(f"처리할 업종 수: {len(run_files)}개")
    for f in run_files:
        print(f"  {os.path.basename(f)}")

    results = []
    failed  = []

    for top10_csv in run_files:
        try:
            res = process_one_industry(top10_csv)
            if res:
                results.append(res)
        except Exception as e:
            name = os.path.basename(top10_csv)
            print(f"\n  [오류] {name} 처리 중 예외 발생: {e}")
            failed.append({"file": name, "error": str(e)})

    print(f"\n\n{'='*70}")
    print(f"전체 처리 완료: 성공 {len(results)}개 / 실패 {len(failed)}개")
    print(f"{'='*70}")

    if failed:
        print(f"\n[실패 목록]")
        for f in failed:
            print(f"  {f['file']} : {f['error']}")
        os.makedirs(SHAP_SAVE_DIR, exist_ok=True)
        pd.DataFrame(failed).to_csv(
            os.path.join(SHAP_SAVE_DIR, "실패_목록.csv"),
            index=False, encoding="utf-8-sig"
        )


if __name__ == "__main__":
    main()

전체 업종 수  : 15개
스킵 업종 수  : 0개
처리할 업종 수: 15개
  M03_음식료품_제조업_Top10 우수모델.csv
  M04_섬유_가죽_신발_제조업_Top10 우수모델.csv
  M08_화학_의약품_고무_플라스틱_제조업_Top10 우수모델.csv
  M10_제1차금속산업_Top10 우수모델.csv
  M11_조립금속제품_제조업_Top10 우수모델.csv
  M12_기타기계장비_제조업_Top10 우수모델.csv
  M13_전자부품_컴퓨터_전기장비_제조업_Top10 우수모델.csv
  M15_운송장비_제조업_Top10 우수모델.csv
  M17_전기_가스_수도사업_Top10 우수모델.csv
  M18_건설업_Top10 우수모델.csv
  M20_숙박_음식점업_Top10 우수모델.csv
  M21_운수_창고업_Top10 우수모델.csv
  M22_정보통신업_Top10 우수모델.csv
  M23_부동산_임대_사업서비스업_Top10 우수모델.csv
  M25_오락_문화_개인서비스업_Top10 우수모델.csv

[업종] M03_음식료품_제조업
  Model=LightGBM | Method=BorderlineSMOTE | FeatureSet=top65_dedup47 | SMOTE_Ratio=0.1
  피처 수: 47개
  리샘플링(BorderlineSMOTE): 6318행 → 6661행 | pos_weight=1.0000
  모델 학습 완료 (LightGBM)
  SHAP 계산 중...
  Permutation Importance 계산 중...
  저장 완료:
    → 중간결과\16_SHAP\M03_음식료품_제조업\feature_analysis_top65_dedup47.csv
    → shap_bar_top65_dedup47.png
    → shap_beeswarm_top65_dedup47.png
    → permutation_bar_top65_dedup47.png

[업종] M04_섬유_가죽_신발_제조업
  Model=LightGBM | Metho

In [3]:
# ============================================================
# 전체 업종 Top10 1위 조합 기반 SHAP + Permutation Importance
# 저장: 중간결과\16_SHAP\{prefix}\
#   - feature_analysis_{FEATURE_SET}.csv
#   - shap_bar_{FEATURE_SET}.png
#   - shap_beeswarm_{FEATURE_SET}.png
#   - permutation_bar_{FEATURE_SET}.png
# ============================================================

import os
import re
import glob
import gc
import platform
import warnings
import numpy as np
import pandas as pd
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("[경고] LightGBM 미설치 → LightGBM 모델 사용 불가")

try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

try:
    from ctgan import CTGAN
    HAS_CTGAN = True
except ImportError:
    HAS_CTGAN = False

warnings.filterwarnings("ignore")

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False


# ============================================================
# 1. 전역 경로 설정
# ============================================================
BASE_DIR       = r'중간결과'
TOP10_FOLDER   = os.path.join(BASE_DIR, r'15_우수모델\Top10')
TRAIN_FOLDER   = os.path.join(BASE_DIR, '12_train')
TEST_FOLDER    = os.path.join(BASE_DIR, '12_test')
FEATURE_FOLDER = os.path.join(BASE_DIR, '13_피처셀렉션')
SHAP_SAVE_DIR  = os.path.join(BASE_DIR, '16_SHAP')

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42


# ============================================================
# 2. 업종 정보 파싱 유틸
# ============================================================

def parse_industry_info(top10_csv_path: str):
    fname = os.path.basename(top10_csv_path)
    match = re.match(r'^(.+?)_Top10', fname)
    if not match:
        raise ValueError(f"파일명 패턴 불일치: {fname}")
    prefix = match.group(1)   # e.g. M03_음식료품_제조업
    return {
        "prefix"         : prefix,
        "train_file"     : os.path.join(TRAIN_FOLDER, f"{prefix}_train.parquet"),
        "test_file"      : os.path.join(TEST_FOLDER,  f"{prefix}_test.parquet"),
        "feature_folder" : os.path.join(FEATURE_FOLDER, prefix),
    }


# ============================================================
# 3. 모델 팩토리
# ============================================================

def make_model(model_name: str, pos_weight: float):
    mn = str(model_name).strip()

    if mn == "XGBoost":
        return XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="aucpr",
            random_state=RANDOM_STATE, verbosity=0,
            scale_pos_weight=pos_weight
        )

    if mn == "LightGBM":
        if not HAS_LGBM:
            raise ImportError("LightGBM 미설치. pip install lightgbm")
        if pos_weight > 1.0:
            return LGBMClassifier(
                n_estimators=300, learning_rate=0.05, max_depth=4,
                subsample=0.8, colsample_bytree=0.8,
                random_state=RANDOM_STATE, verbose=-1,
                scale_pos_weight=pos_weight
            )
        return LGBMClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8,
            random_state=RANDOM_STATE, verbose=-1
        )

    if mn == "RandomForest":
        if pos_weight > 1.0:
            return RandomForestClassifier(
                n_estimators=300, max_depth=4, min_samples_leaf=5,
                random_state=RANDOM_STATE, n_jobs=-1,
                class_weight="balanced"
            )
        return RandomForestClassifier(
            n_estimators=300, max_depth=4, min_samples_leaf=5,
            random_state=RANDOM_STATE, n_jobs=-1
        )

    if mn == "LogisticRegression":
        if pos_weight > 1.0:
            return LogisticRegression(
                penalty="l2", C=1.0, solver="lbfgs",
                max_iter=1000, random_state=RANDOM_STATE,
                class_weight="balanced"
            )
        return LogisticRegression(
            penalty="l2", C=1.0, solver="lbfgs",
            max_iter=1000, random_state=RANDOM_STATE
        )

    raise ValueError(f"지원하지 않는 모델명: {model_name}")


# ============================================================
# 4. 불균형 처리 함수
# ============================================================

def _parse_smote_ratio(smote_ratio):
    try:
        if smote_ratio is None:
            return None
        s = str(smote_ratio).strip()
        if s in ("", "-", "nan", "None"):
            return None
        return float(s)
    except (TypeError, ValueError):
        return None


def _ctgan_resample(X, y, ratio):
    if not HAS_CTGAN:
        raise ImportError("CTGAN 방식을 사용하려면 'pip install ctgan'이 필요합니다.")
    if not isinstance(X, pd.DataFrame):
        raise TypeError("_ctgan_resample: X는 pd.DataFrame이어야 합니다.")

    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)

    n_minority = int((y == 1).sum())
    n_majority = int((y == 0).sum())

    if n_minority < 5:
        print(f"    [CTGAN 경고] 소수 클래스({n_minority}개) 너무 적어 합성 건너뜁니다.")
        return X, y

    target_minority = int(n_majority * ratio) if ratio is not None else n_majority
    n_synth = target_minority - n_minority

    if n_synth <= 0:
        print(f"    [CTGAN] 현재 소수({n_minority}) >= 목표({target_minority}), 합성 불필요.")
        return X, y

    print(f"    [CTGAN] 소수={n_minority}, 다수={n_majority}, ratio={ratio}, 합성수={n_synth}")
    minority_df = X[y == 1].copy().reset_index(drop=True)

    try:
        ctgan = CTGAN(epochs=300, verbose=False)
    except TypeError:
        ctgan = CTGAN(epochs=300)

    ctgan.fit(minority_df)
    synth = ctgan.sample(n_synth).reindex(columns=X.columns)

    X_res = pd.concat([X, synth], ignore_index=True)
    y_res = pd.concat([y, pd.Series([1] * n_synth)], ignore_index=True)
    print(f"    [CTGAN] 완료 → 정상={int((y_res==0).sum())}, 부실={int((y_res==1).sum())}")
    return X_res, y_res


def apply_resampling(X, y, method, smote_ratio):
    method_l = str(method).strip().lower()
    ratio    = _parse_smote_ratio(smote_ratio)

    if method_l == "classweight":
        pos_weight = (y == 0).sum() / (y == 1).sum()
        return X, y, pos_weight

    if method_l in ("none", "", "-", "nan"):
        return X, y, 1.0

    if "borderline" in method_l:
        if not HAS_IMBLEARN:
            raise ImportError("pip install imbalanced-learn")
        sampler = BorderlineSMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if method_l == "smote":
        if not HAS_IMBLEARN:
            raise ImportError("pip install imbalanced-learn")
        sampler = SMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if "ctgan" in method_l:
        X_res, y_res = _ctgan_resample(X, y, ratio)
        return X_res, y_res, 1.0

    print(f"  [경고] 알 수 없는 Method='{method}' → ClassWeight로 처리합니다.")
    pos_weight = (y == 0).sum() / (y == 1).sum()
    return X, y, pos_weight


# ============================================================
# 5. SHAP Explainer 선택 (모델 종류별)
# ============================================================

def get_shap_values(model, model_name: str, X: pd.DataFrame):
    """
    - Tree 계열 (XGBoost, LightGBM, RandomForest): TreeExplainer
    - 그 외 (LogisticRegression 등): LinearExplainer (또는 KernelExplainer fallback)
    반환: shap_values (2D array, shape=[n_samples, n_features])
    """
    mn = str(model_name).strip()

    if mn in ("XGBoost", "LightGBM", "RandomForest"):
        explainer = shap.TreeExplainer(model)

        # check_additivity=False: 검증 계산 생략 → 메모리/속도 절약
        shap_values = explainer.shap_values(X, check_additivity=False)

        # ── 반환 형태 디버깅 출력
        if isinstance(shap_values, list):
            print(f"    [SHAP debug] list 길이={len(shap_values)}, "
                  f"element[0] shape={np.array(shap_values[0]).shape}")
        else:
            print(f"    [SHAP debug] array shape={np.array(shap_values).shape}")

        # ── 반환 형태 정규화 → 항상 2D (n_samples, n_features)로 만들기
        # case 1: list → 양성 클래스(index 1) 선택 (이진분류: list 길이 2)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]

        shap_values = np.array(shap_values)

        # case 2: 3D (n_samples, n_features, n_classes) → [:, :, 1]
        if shap_values.ndim == 3:
            shap_values = shap_values[:, :, 1]

        # case 3: 1D → 2D reshape
        if shap_values.ndim == 1:
            shap_values = shap_values.reshape(1, -1)

        print(f"    [SHAP debug] 정규화 후 shape={shap_values.shape}")

        if shap_values.ndim != 2:
            raise ValueError(f"예상치 못한 shap_values shape: {shap_values.shape}")
        return shap_values

    # LogisticRegression → LinearExplainer
    if mn == "LogisticRegression":
        explainer   = shap.LinearExplainer(model, X)
        shap_values = explainer.shap_values(X)

        if isinstance(shap_values, list):
            shap_values = shap_values[1]

        shap_values = np.array(shap_values)
        if shap_values.ndim == 3:
            shap_values = shap_values[:, :, 1]

        return shap_values

    # fallback: KernelExplainer (느리므로 샘플 200개 제한)
    print(f"  [SHAP] 알 수 없는 모델({mn}) → KernelExplainer 사용 (샘플 200개)")
    bg = shap.sample(X, 100)
    explainer   = shap.KernelExplainer(model.predict_proba, bg)
    shap_values = explainer.shap_values(X.sample(min(200, len(X)),
                                                   random_state=RANDOM_STATE))
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    return shap_values


# ============================================================
# 6. 피처 카테고리 매핑
#    - 베이스 피처명(접미사 제거 전)으로 카테고리 정의
#    - _diff / _ratio / _diff_industry / _ratio_industry 등
#      모든 변형 컬럼은 get_category() 함수에서 접미사 제거 후 조회
# ============================================================

# 베이스 피처명 → 카테고리 (접미사 없는 원형 기준)
FEATURE_CATEGORY_BASE = {
    # ── 안정성 (Solvency) ──────────────────────────────────
    "부채비율"              : "안정성 (Solvency)",
    "총부채비율"            : "안정성 (Solvency)",
    "장기부채비율"          : "안정성 (Solvency)",
    "장기부채의존도"        : "안정성 (Solvency)",
    "차입금의존도"          : "안정성 (Solvency)",
    "순차입금비율"          : "안정성 (Solvency)",
    "금융부채비율"          : "안정성 (Solvency)",
    "자기자본비율"          : "안정성 (Solvency)",
    "유보율"                : "안정성 (Solvency)",
    "자본잠식률"            : "안정성 (Solvency)",
    "유동비율"              : "안정성 (Solvency)",
    "당좌비율"              : "안정성 (Solvency)",
    "당좌비율_추정"         : "안정성 (Solvency)",
    "현금비율"              : "안정성 (Solvency)",
    "순운전자본비율"        : "안정성 (Solvency)",
    "순운전자본대총자본"    : "안정성 (Solvency)",
    "비유동비율"            : "안정성 (Solvency)",
    "비유동장기적합률"      : "안정성 (Solvency)",
    "유형자산부채비율"      : "안정성 (Solvency)",
    "부채비율변화"          : "안정성 (Solvency)",
    "유동비율변화"          : "안정성 (Solvency)",

    # ── 수익성 (Profitability) ─────────────────────────────
    "ROA"                   : "수익성 (Profitability)",
    "ROE"                   : "수익성 (Profitability)",
    "ROIC"                  : "수익성 (Profitability)",
    "총자본영업이익률"      : "수익성 (Profitability)",
    "매출총이익률"          : "수익성 (Profitability)",
    "영업이익률"            : "수익성 (Profitability)",
    "순이익률"              : "수익성 (Profitability)",
    "EBITDA마진"            : "수익성 (Profitability)",
    "현금ROA"               : "수익성 (Profitability)",
    "현금ROE"               : "수익성 (Profitability)",
    "매출원가율"            : "수익성 (Profitability)",
    "판관비율"              : "수익성 (Profitability)",
    "금융비용부담률"        : "수익성 (Profitability)",
    "ROA변화"               : "수익성 (Profitability)",
    "영업이익률변화"        : "수익성 (Profitability)",
    "매출액순이익률"        : "수익성 (Profitability)",
    "매출액영업이익률"      : "수익성 (Profitability)",
    "EBIT대매출액"          : "수익성 (Profitability)",
    "EBITDA대매출액"        : "수익성 (Profitability)",
    "매출원가대매출액"      : "수익성 (Profitability)",
    "금융비용대매출액"      : "수익성 (Profitability)",
    "사내유보율"            : "수익성 (Profitability)",

    # ── 성장성 (Growth) ────────────────────────────────────
    "매출액증가율"          : "성장성 (Growth)",
    "영업이익증가율"        : "성장성 (Growth)",
    "순이익증가율"          : "성장성 (Growth)",
    "EBITDA증가율"          : "성장성 (Growth)",
    "총자산증가율"          : "성장성 (Growth)",
    "유형자산증가율"        : "성장성 (Growth)",
    "자기자본증가율"        : "성장성 (Growth)",
    "부채증가율"            : "성장성 (Growth)",
    "영업현금흐름증가율"    : "성장성 (Growth)",
    "FCF증가율"             : "성장성 (Growth)",

    # ── 활동성 (Activity) ──────────────────────────────────
    "총자산회전율"          : "활동성 (Activity)",
    "유동자산회전율"        : "활동성 (Activity)",
    "비유동자산회전율"      : "활동성 (Activity)",
    "유형자산회전율"        : "활동성 (Activity)",
    "자기자본회전율"        : "활동성 (Activity)",
    "투하자본회전율"        : "활동성 (Activity)",
    "매출채권회전율"        : "활동성 (Activity)",
    "매출채권회수기간"      : "활동성 (Activity)",
    "재고자산회전율"        : "활동성 (Activity)",
    "재고자산보유기간"      : "활동성 (Activity)",
    "매입채무회전율"        : "활동성 (Activity)",
    "매입채무지급기간"      : "활동성 (Activity)",
    "현금전환주기_CCC"      : "활동성 (Activity)",
    "순운전자본회전율"      : "활동성 (Activity)",

    # ── 현금흐름 (Cash Flow) ───────────────────────────────
    "영업현금흐름비율"      : "현금흐름 (Cash Flow)",
    "영업CF_유동부채"       : "현금흐름 (Cash Flow)",
    "영업CF_총부채"         : "현금흐름 (Cash Flow)",
    "FCF_총자산"            : "현금흐름 (Cash Flow)",
    "감가상각비율"          : "현금흐름 (Cash Flow)",

    # ── 기타 (Other) ───────────────────────────────────────
    "업력"                  : "기타 (Other)",
    "빅4감사"               : "기타 (Other)",
    "유형자산비율"          : "기타 (Other)",
    "종업원"                : "기타 (Other)",
}

# 접미사 제거 순서 (긴 것부터 먼저 제거해야 오매칭 방지)
_SUFFIXES = [
    "_diff_industry",
    "_ratio_industry",
    "_diff",
    "_ratio",
]

def get_category(feature_name: str) -> str:
    """
    피처명에서 접미사(_diff/_ratio/_diff_industry/_ratio_industry)를
    순서대로 제거한 뒤 FEATURE_CATEGORY_BASE에서 카테고리를 찾는다.
    직접 매핑이 없으면 접미사 제거 후 재시도.
    """
    # 1) 완전 일치 우선
    if feature_name in FEATURE_CATEGORY_BASE:
        return FEATURE_CATEGORY_BASE[feature_name]

    # 2) 접미사 제거 후 재시도
    base = feature_name
    for sfx in _SUFFIXES:
        if base.endswith(sfx):
            base = base[: -len(sfx)]
            break   # 한 번만 제거

    if base in FEATURE_CATEGORY_BASE:
        return FEATURE_CATEGORY_BASE[base]

    return "미분류"


CATEGORY_ORDER = {
    "안정성 (Solvency)"      : 1,
    "수익성 (Profitability)" : 2,
    "성장성 (Growth)"        : 3,
    "활동성 (Activity)"      : 4,
    "현금흐름 (Cash Flow)"   : 5,
    "기타 (Other)"           : 6,
    "미분류"                 : 7,
}


# ============================================================
# 7. 단일 업종 처리 함수
# ============================================================

def process_one_industry(top10_csv_path: str):
    info   = parse_industry_info(top10_csv_path)
    prefix = info["prefix"]

    print(f"\n{'='*70}")
    print(f"[업종] {prefix}")
    print(f"{'='*70}")

    # ── Top10 1위 행 로드
    top10 = pd.read_csv(top10_csv_path, index_col=0)
    if len(top10) == 0:
        print(f"  [경고] 파일이 비어있습니다. skip.")
        return None

    row          = top10.iloc[0]
    FEATURE_SET  = row["FeatureSet"]
    FEATURE_FILE = row["FeatureFile"]
    METHOD       = row["Method"]
    SMOTE_RATIO  = row["SMOTE_Ratio"]
    MODEL_NAME   = row["Model"]

    print(f"  Model={MODEL_NAME} | Method={METHOD} | "
          f"FeatureSet={FEATURE_SET} | SMOTE_Ratio={SMOTE_RATIO}")

    # ── 저장 폴더
    save_sub = os.path.join(SHAP_SAVE_DIR, prefix)
    os.makedirs(save_sub, exist_ok=True)

    # ── 파일 존재 확인
    feature_path = os.path.join(info["feature_folder"], FEATURE_FILE)
    for label, path in [("피처 파일", feature_path),
                         ("Train",    info["train_file"]),
                         ("Test",     info["test_file"])]:
        if not os.path.exists(path):
            print(f"  [경고] {label} 없음: {path} → skip")
            return None

    # ── 데이터 로드
    train_full = pd.read_parquet(info["train_file"])
    test       = pd.read_parquet(info["test_file"])
    y_train    = train_full[TARGET_COL]
    y_test     = test[TARGET_COL]

    feat_df      = pd.read_csv(feature_path)
    col_key      = "feature" if "feature" in feat_df.columns else feat_df.columns[0]
    use_features = [f for f in feat_df[col_key].tolist() if f in train_full.columns]
    print(f"  피처 수: {len(use_features)}개")

    # ── 결측치 처리
    imputer     = SimpleImputer(strategy="median")
    X_train_imp = pd.DataFrame(
        imputer.fit_transform(train_full[use_features]), columns=use_features
    )
    X_test_imp  = pd.DataFrame(
        imputer.transform(test[use_features]), columns=use_features
    )

    # ── 리샘플링 + 모델 학습
    X_res, y_res, pos_weight = apply_resampling(X_train_imp, y_train, METHOD, SMOTE_RATIO)
    print(f"  리샘플링({METHOD}): {len(X_train_imp)}행 → {len(X_res)}행 | pos_weight={pos_weight:.4f}")

    model = make_model(MODEL_NAME, pos_weight)
    model.fit(X_res, y_res)
    print(f"  모델 학습 완료 ({MODEL_NAME})")

    # ── SHAP 계산 (메모리 절약: test 샘플 최대 1000개로 제한)
    SHAP_MAX_SAMPLES = 1000
    if len(X_test_imp) > SHAP_MAX_SAMPLES:
        X_shap = X_test_imp.sample(SHAP_MAX_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)
        print(f"  SHAP 계산 중... (test {len(X_test_imp)}행 → {SHAP_MAX_SAMPLES}행 샘플링)")
    else:
        X_shap = X_test_imp.reset_index(drop=True)
        print(f"  SHAP 계산 중... ({len(X_shap)}행)")

    shap_values = get_shap_values(model, MODEL_NAME, X_shap)

    shap_importance = pd.DataFrame({
        "Feature"       : use_features,
        "mean_abs_SHAP" : np.abs(shap_values).mean(axis=0)
    }).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)
    shap_importance["Rank"] = shap_importance.index + 1

    # ── PNG 1: SHAP Bar
    fig = plt.figure(figsize=(10, max(6, len(use_features) * 0.3)))
    shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False, max_display=20)
    plt.title(f"SHAP Global Feature Importance (Bar)\n{prefix} | {FEATURE_SET} ({METHOD})",
              fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(save_sub, f"shap_bar_{FEATURE_SET}.png"),
                dpi=150, bbox_inches="tight")
    plt.close("all")

    # ── PNG 2: SHAP Beeswarm
    fig = plt.figure(figsize=(10, max(6, len(use_features) * 0.3)))
    shap.summary_plot(shap_values, X_shap, plot_type="dot", show=False, max_display=20)
    plt.title(f"SHAP Beeswarm Plot\n{prefix} | {FEATURE_SET} ({METHOD})", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(save_sub, f"shap_beeswarm_{FEATURE_SET}.png"),
                dpi=150, bbox_inches="tight")
    plt.close("all")

    # shap_values 메모리 해제
    del shap_values

    # ── Permutation Importance (test 전체 사용, n_jobs=1 → Jupyter 크래시 방지)
    print(f"  Permutation Importance 계산 중...")
    perm_result = permutation_importance(
        model, X_test_imp, y_test,
        n_repeats=10, random_state=RANDOM_STATE,   # 30 → 10 (속도/안정성 균형)
        scoring="average_precision", n_jobs=1       # -1 → 1 (Jupyter 멀티프로세싱 크래시 방지)
    )
    perm_df = pd.DataFrame({
        "Feature"   : use_features,
        "Perm_Mean" : perm_result.importances_mean,
        "Perm_Std"  : perm_result.importances_std,
    }).sort_values("Perm_Mean", ascending=False).reset_index(drop=True)
    perm_df["Rank"] = perm_df.index + 1

    # ── PNG 3: Permutation Bar
    top_n    = min(20, len(use_features))
    perm_top = perm_df.head(top_n).sort_values("Perm_Mean", ascending=True)
    fig, ax  = plt.subplots(figsize=(10, max(6, top_n * 0.4)))
    ax.barh(
        perm_top["Feature"], perm_top["Perm_Mean"],
        xerr=perm_top["Perm_Std"],
        color="#2F6EBA", alpha=0.8,
        error_kw=dict(ecolor="#555555", capsize=3)
    )
    ax.axvline(0, color="red", linestyle="--", linewidth=1.0)
    ax.set_xlabel("Mean decrease in PR_AUC", fontsize=10)
    ax.set_title(f"Permutation Importance (Top {top_n})\n{prefix} | {FEATURE_SET} ({METHOD})",
                 fontsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(save_sub, f"permutation_bar_{FEATURE_SET}.png"),
                dpi=150, bbox_inches="tight")
    plt.close()

    # ── feature_analysis CSV 생성
    n_features = len(shap_importance)

    df = shap_importance[["Rank", "Feature", "mean_abs_SHAP"]].rename(
        columns={"Rank": "SHAP_Rank"}
    ).merge(
        perm_df[["Feature", "Perm_Mean", "Perm_Std", "Rank"]].rename(
            columns={"Rank": "Perm_Rank"}
        ),
        on="Feature", how="inner"
    )

    df["SHAP_Rank_Norm"] = (df["SHAP_Rank"] - 1) / max(n_features - 1, 1)
    df["Perm_Rank_Norm"] = (df["Perm_Rank"] - 1) / max(n_features - 1, 1)
    df["Rank_Diff"]      = (df["SHAP_Rank"] - df["Perm_Rank"]).abs()
    df["Combined_Score"] = (df["SHAP_Rank_Norm"] + df["Perm_Rank_Norm"]) / 2
    df["Combined_Rank"]  = df["Combined_Score"].rank(method="min").astype(int)

    top_n_cls    = max(3, int(n_features * 0.3))
    shap_top_set = set(df.nsmallest(top_n_cls, "SHAP_Rank")["Feature"])
    perm_top_set = set(df.nsmallest(top_n_cls, "Perm_Rank")["Feature"])

    def classify(r):
        in_shap = r["Feature"] in shap_top_set
        in_perm = r["Feature"] in perm_top_set
        if   in_shap and in_perm:     return "★ 핵심피처 (SHAP+Perm 모두 높음)"
        elif in_shap and not in_perm: return "△ 대체가능 (SHAP 높음, Perm 낮음)"
        elif not in_shap and in_perm: return "▲ 상호작용 (Perm 높음, SHAP 낮음)"
        else:                         return "- 일반피처"

    def rank_diff_level(diff):
        if diff <= 3:   return "일치"
        elif diff <= 8: return "소폭 불일치"
        else:           return "대폭 불일치"

    df["Feature_Type"]   = df.apply(classify, axis=1)
    df["Consistency"]    = df["Rank_Diff"].apply(rank_diff_level)
    df["Category"]       = df["Feature"].apply(get_category)
    df["Category_Order"] = df["Category"].map(CATEGORY_ORDER)

    # 미분류 피처 있으면 출력 (디버깅용)
    unclassified = df[df["Category"] == "미분류"]["Feature"].tolist()
    if unclassified:
        print(f"  [카테고리 미분류 피처 {len(unclassified)}개]: {unclassified}")

    df = df[[
        "Combined_Rank", "Feature", "Category", "Feature_Type",
        "SHAP_Rank", "mean_abs_SHAP",
        "Perm_Rank", "Perm_Mean", "Perm_Std",
        "Rank_Diff", "Consistency",
        "Combined_Score", "Category_Order",
    ]].sort_values("Combined_Rank").reset_index(drop=True)

    csv_path = os.path.join(save_sub, f"feature_analysis_{FEATURE_SET}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"  저장 완료:")
    print(f"    → {csv_path}")
    print(f"    → shap_bar_{FEATURE_SET}.png")
    print(f"    → shap_beeswarm_{FEATURE_SET}.png")
    print(f"    → permutation_bar_{FEATURE_SET}.png")

    # 메모리 명시적 해제
    del train_full, test, X_train_imp, X_test_imp, X_shap, X_res, model
    gc.collect()

    return {"Industry": prefix, "FeatureSet": FEATURE_SET, "Model": MODEL_NAME,
            "Method": METHOD, "N_Features": n_features}


# ============================================================
# 8. 전체 Top10 파일 순회 실행
# ============================================================

def main():
    top10_files = sorted(glob.glob(os.path.join(TOP10_FOLDER, "*.csv")))
    if not top10_files:
        print(f"[오류] Top10 파일이 없습니다: {TOP10_FOLDER}")
        return

    # ── 재실행할 업종만 지정 (빈 리스트면 전체 실행)
    RETRY_ONLY = [
        "M10_제1차금속산업",
        "M12_기타기계장비_제조업",
        "M13_전자부품_컴퓨터_전기장비_제조업",
        "M18_건설업",
        "M21_운수_창고업",
        "M25_오락_문화_개인서비스업",
    ]

    if RETRY_ONLY:
        run_files = [f for f in top10_files
                     if any(kw in os.path.basename(f) for kw in RETRY_ONLY)]
        print(f"[재실행 모드] 대상 업종 수: {len(run_files)}개")
    else:
        run_files = top10_files
        print(f"[전체 실행 모드] 업종 수: {len(run_files)}개")

    for f in run_files:
        print(f"  {os.path.basename(f)}")

    results = []
    failed  = []

    for top10_csv in run_files:
        try:
            res = process_one_industry(top10_csv)
            if res:
                results.append(res)
        except Exception as e:
            name = os.path.basename(top10_csv)
            print(f"\n  [오류] {name} 처리 중 예외 발생: {e}")
            failed.append({"file": name, "error": str(e)})

    print(f"\n\n{'='*70}")
    print(f"전체 처리 완료: 성공 {len(results)}개 / 실패 {len(failed)}개")
    print(f"{'='*70}")

    if failed:
        print(f"\n[실패 목록]")
        for f in failed:
            print(f"  {f['file']} : {f['error']}")
        os.makedirs(SHAP_SAVE_DIR, exist_ok=True)
        pd.DataFrame(failed).to_csv(
            os.path.join(SHAP_SAVE_DIR, "실패_목록.csv"),
            index=False, encoding="utf-8-sig"
        )


if __name__ == "__main__":
    main()

[재실행 모드] 대상 업종 수: 6개
  M10_제1차금속산업_Top10 우수모델.csv
  M12_기타기계장비_제조업_Top10 우수모델.csv
  M13_전자부품_컴퓨터_전기장비_제조업_Top10 우수모델.csv
  M18_건설업_Top10 우수모델.csv
  M21_운수_창고업_Top10 우수모델.csv
  M25_오락_문화_개인서비스업_Top10 우수모델.csv

[업종] M10_제1차금속산업
  Model=RandomForest | Method=CTGAN | FeatureSet=top65_dedup44 | SMOTE_Ratio=0.2
  피처 수: 44개
    [CTGAN] 소수=289, 다수=6242, ratio=0.2, 합성수=959
    [CTGAN] 완료 → 정상=6242, 부실=1248
  리샘플링(CTGAN): 6531행 → 7490행 | pos_weight=1.0000
  모델 학습 완료 (RandomForest)
  SHAP 계산 중... (test 1954행 → 1000행 샘플링)
    [SHAP debug] array shape=(1000, 44, 2)
    [SHAP debug] 정규화 후 shape=(1000, 44)
  Permutation Importance 계산 중...
  저장 완료:
    → 중간결과\16_SHAP\M10_제1차금속산업\feature_analysis_top65_dedup44.csv
    → shap_bar_top65_dedup44.png
    → shap_beeswarm_top65_dedup44.png
    → permutation_bar_top65_dedup44.png

[업종] M12_기타기계장비_제조업
  Model=RandomForest | Method=ClassWeight | FeatureSet=top50_dedup38 | SMOTE_Ratio=-
  피처 수: 37개
  리샘플링(ClassWeight): 14324행 → 14324행 | pos_weight=24.9964
  모델 학

# pd 변화율

In [ ]:
# ============================================================
# 전체 업종 기업 부실 확률 변화(차이값) 분석
# 입력: 중간결과\15_우수모델\2014_2024_PD\{prefix}_{FEATURE_SET}_2014_2024_PD_데이터.csv
# 출력: 중간결과\21_PD변화율\{prefix}_{FEATURE_SET}_2015-2024_기업_부실확률_차이값.csv
# ============================================================

import os
import re
import glob
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# ============================================================
# 1. 전역 경로 설정
# ============================================================
BASE_DIR     = r'중간결과'
TOP10_FOLDER = os.path.join(BASE_DIR, r'15_우수모델\Top10')
PD_FOLDER    = os.path.join(BASE_DIR, r'15_우수모델\2014_2024_PD')
SAVE_DIR     = os.path.join(BASE_DIR, '21_PD변화율')
os.makedirs(SAVE_DIR, exist_ok=True)

YEAR_COL    = "회계년도"
COMPANY_COL = "회사명"
ID_COL      = "사업자등록번호"

TARGET_YEARS = list(range(2014, 2025))   # 2014~2024, 총 11개 연도


# ============================================================
# 2. 업종 정보 파싱 유틸
# ============================================================

def parse_industry_info(top10_csv_path):
    fname = os.path.basename(top10_csv_path)
    match = re.match(r'^(.+?)_Top10', fname)
    if not match:
        raise ValueError(f"파일명 패턴 불일치: {fname}")
    return match.group(1)   # e.g. M19_도매_소매업


def get_pd_path(prefix, feature_set):
    """
    중간결과\15_우수모델\2014_2024_PD\{prefix}_{feature_set}_2014_2024_PD_데이터.csv
    """
    fname = f"{prefix}_{feature_set}_2014_2024_PD_데이터.csv"
    return os.path.join(PD_FOLDER, fname)


# ============================================================
# 3. 위험 신호 분류 함수
# ============================================================

def make_risk_signal_fn(threshold, last3_diffs, last2_diffs, last_year):
    def risk_signal(row):
        p_last = row[f"prob_{last_year}"]

        if pd.isna(p_last):
            return "데이터없음(최종연도)"

        d_a, d_b, d_c = (row[c] for c in last3_diffs)
        d_b2, d_c2    = (row[c] for c in last2_diffs)

        accel_3  = d_a > 0 and d_b > 0 and d_c > 0 and d_a < d_b < d_c
        rising_3 = d_a > 0 and d_b > 0 and d_c > 0
        accel_2  = d_b2 > 0 and d_c2 > 0 and d_b2 < d_c2
        rising_2 = d_b2 > 0 and d_c2 > 0

        if accel_3:
            if p_last >= threshold:         return "위험 (가속+임계초과)"
            elif p_last >= threshold * 0.7: return "주의 (가속+임계근접)"
            else:                           return "관찰 (가속)"

        if rising_3:
            if p_last >= threshold:         return "위험 (3년연속상승+임계초과)"
            elif p_last >= threshold * 0.7: return "주의 (3년연속상승+임계근접)"
            else:                           return "관찰 (3년연속상승)"

        if accel_2:
            if p_last >= threshold:         return "위험 (최근가속+임계초과)"
            elif p_last >= threshold * 0.7: return "주의 (최근가속+임계근접)"
            else:                           return "관찰 (최근가속)"

        if rising_2:
            if p_last >= threshold:         return "위험 (최근2년상승+임계초과)"
            elif p_last >= threshold * 0.7: return "주의 (최근2년상승+임계근접)"
            else:                           return "관찰 (최근2년상승)"

        if pd.notna(d_c2) and d_c2 > 0.05:
            if p_last >= threshold:         return "위험 (최근급등+임계초과)"
            else:                           return "주의 (최근급등)"

        if p_last >= threshold:             return "위험 (임계초과)"
        if p_last < threshold * 0.5:        return "안정"
        return "보통"

    return risk_signal


# ============================================================
# 4. 단일 업종 처리 함수
# ============================================================

def process_one_industry(top10_csv_path):
    prefix = parse_industry_info(top10_csv_path)

    print(f"\n{'='*70}")
    print(f"[업종] {prefix}")
    print(f"{'='*70}")

    # ── Top10 1위 행에서 FEATURE_SET, THRESHOLD 가져오기
    top10 = pd.read_csv(top10_csv_path, index_col=0)
    if len(top10) == 0:
        print(f"  [경고] Top10 파일이 비어있습니다. skip.")
        return None

    row         = top10.iloc[0]
    FEATURE_SET = row["FeatureSet"]

    # Threshold: Top10 summary에 컬럼이 있으면 사용, 없으면 기본값 0.44
    if "Threshold" in row.index and pd.notna(row.get("Threshold", None)):
        THRESHOLD = float(row["Threshold"])
    else:
        THRESHOLD = 0.44
        print(f"  [참고] Threshold 컬럼 없음 → 기본값 {THRESHOLD} 사용")

    print(f"  FeatureSet={FEATURE_SET} | Threshold={THRESHOLD}")

    # ── PD 데이터 파일 경로 확인
    pd_path = get_pd_path(prefix, FEATURE_SET)
    if not os.path.exists(pd_path):
        print(f"  [경고] PD 데이터 파일 없음: {pd_path} → skip")
        return None

    # ── 데이터 로드
    pd_df = pd.read_csv(pd_path, encoding="utf-8-sig")
    pd_df[YEAR_COL] = pd_df[YEAR_COL].astype(int)

    print(f"  PD 데이터: {len(pd_df)}행 | "
          f"연도 {pd_df[YEAR_COL].min()}~{pd_df[YEAR_COL].max()} | "
          f"기업 수 {pd_df[ID_COL].nunique()}개")

    # ── 분석 대상 필터 (2014~2024)
    analysis_data = pd_df[pd_df[YEAR_COL].isin(TARGET_YEARS)].copy()

    # ── 연도별 피벗
    pivot = analysis_data.pivot_table(
        index=[ID_COL, COMPANY_COL],
        columns=YEAR_COL,
        values="y_prob"
    ).reset_index()
    pivot.columns.name = None

    rename_map = {yr: f"prob_{yr}" for yr in TARGET_YEARS}
    pivot = pivot.rename(columns=rename_map)

    prob_cols = [f"prob_{yr}" for yr in TARGET_YEARS]
    for col in prob_cols:
        if col not in pivot.columns:
            pivot[col] = np.nan

    # ── 차이값 계산 (현재 - 이전)
    diff_cols = []
    for prev_yr, curr_yr in zip(TARGET_YEARS[:-1], TARGET_YEARS[1:]):
        col_name = f"diff_{prev_yr}_{curr_yr}"
        pivot[col_name] = (
            pivot[f"prob_{curr_yr}"] - pivot[f"prob_{prev_yr}"]
        ).round(4)
        diff_cols.append(col_name)

    for col in diff_cols:
        pivot[col] = pivot[col].fillna(0)

    # ── 위험 신호 분류
    LAST_YEAR   = TARGET_YEARS[-1]       # 2024
    last3_diffs = diff_cols[-3:]         # diff_2021_2022, diff_2022_2023, diff_2023_2024
    last2_diffs = diff_cols[-2:]         # diff_2022_2023, diff_2023_2024

    risk_fn = make_risk_signal_fn(THRESHOLD, last3_diffs, last2_diffs, LAST_YEAR)
    pivot["risk_signal"] = pivot.apply(risk_fn, axis=1)

    # ── 위험 점수 (0~100)
    rank_d_a_col  = f"rank_{last3_diffs[0]}"
    rank_d_b_col  = f"rank_{last3_diffs[1]}"
    rank_d_c_col  = f"rank_{last3_diffs[2]}"
    rank_prob_col = f"rank_prob_{LAST_YEAR}"

    pivot[rank_d_a_col]  = pivot[last3_diffs[0]].rank(pct=True) * 100
    pivot[rank_d_b_col]  = pivot[last3_diffs[1]].rank(pct=True) * 100
    pivot[rank_d_c_col]  = pivot[last3_diffs[2]].rank(pct=True) * 100
    pivot[rank_prob_col] = pivot[f"prob_{LAST_YEAR}"].rank(pct=True) * 100

    pivot["risk_score"] = (
        (
            pivot[rank_d_a_col] * 0.2 +
            pivot[rank_d_b_col] * 0.3 +
            pivot[rank_d_c_col] * 0.5
        ) * 0.5 +
        pivot[rank_prob_col] * 0.5
    ).round(2)

    # ── 결과 컬럼 정리
    result_df = pivot[[
        ID_COL, COMPANY_COL,
        *prob_cols,
        *diff_cols,
        "risk_score",
        "risk_signal",
    ]].copy()

    for col in prob_cols:
        result_df[col] = result_df[col].round(4)
    result_df["risk_score"] = result_df["risk_score"].round(2)

    result_df = result_df.sort_values(
        "risk_score", ascending=False, na_position="last"
    ).reset_index(drop=True)

    # ── 저장
    save_fname = f"{prefix}_{FEATURE_SET}_2015-2024_기업_부실확률_차이값.csv"
    save_path  = os.path.join(SAVE_DIR, save_fname)
    result_df.to_csv(save_path, index=False, encoding="utf-8-sig")

    # ── 요약 출력
    print(f"  저장 완료 → {save_path}  ({len(result_df)}행)")
    print(f"  [위험 신호 분포]")
    for sig, cnt in result_df["risk_signal"].value_counts(dropna=False).items():
        print(f"    {sig} : {cnt}개")

    return {
        "Industry"    : prefix,
        "FeatureSet"  : FEATURE_SET,
        "Threshold"   : THRESHOLD,
        "N_Companies" : len(result_df),
    }


# ============================================================
# 5. 전체 Top10 파일 순회 실행
# ============================================================

def main():
    top10_files = sorted(glob.glob(os.path.join(TOP10_FOLDER, "*.csv")))
    if not top10_files:
        print(f"[오류] Top10 파일이 없습니다: {TOP10_FOLDER}")
        return

    print(f"전체 업종 수: {len(top10_files)}개")
    for f in top10_files:
        print(f"  {os.path.basename(f)}")

    results = []
    failed  = []

    for top10_csv in top10_files:
        try:
            res = process_one_industry(top10_csv)
            if res:
                results.append(res)
        except Exception as e:
            name = os.path.basename(top10_csv)
            print(f"\n  [오류] {name} 처리 중 예외 발생: {e}")
            failed.append({"file": name, "error": str(e)})

    # ── 전체 요약
    print(f"\n\n{'='*70}")
    print(f"전체 처리 완료: 성공 {len(results)}개 / 실패 {len(failed)}개")
    print(f"{'='*70}")

    if results:
        pd.DataFrame(results).to_csv(
            os.path.join(SAVE_DIR, "전체_업종_PD변화율_처리요약.csv"),
            index=False, encoding="utf-8-sig"
        )

    if failed:
        print(f"\n[실패 목록]")
        for f in failed:
            print(f"  {f['file']} : {f['error']}")
        pd.DataFrame(failed).to_csv(
            os.path.join(SAVE_DIR, "실패_목록.csv"),
            index=False, encoding="utf-8-sig"
        )


if __name__ == "__main__":
    main()

<>:47: SyntaxWarning: invalid escape sequence '\{'
<>:47: SyntaxWarning: invalid escape sequence '\{'
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22140\3286264004.py:47: SyntaxWarning: invalid escape sequence '\{'
  중간결과\15_우수모델\2014_2024_PD\{prefix}_{feature_set}_2014_2024_PD_데이터.csv


전체 업종 수: 15개
  M03_음식료품_제조업_Top10 우수모델.csv
  M04_섬유_가죽_신발_제조업_Top10 우수모델.csv
  M08_화학_의약품_고무_플라스틱_제조업_Top10 우수모델.csv
  M10_제1차금속산업_Top10 우수모델.csv
  M11_조립금속제품_제조업_Top10 우수모델.csv
  M12_기타기계장비_제조업_Top10 우수모델.csv
  M13_전자부품_컴퓨터_전기장비_제조업_Top10 우수모델.csv
  M15_운송장비_제조업_Top10 우수모델.csv
  M17_전기_가스_수도사업_Top10 우수모델.csv
  M18_건설업_Top10 우수모델.csv
  M20_숙박_음식점업_Top10 우수모델.csv
  M21_운수_창고업_Top10 우수모델.csv
  M22_정보통신업_Top10 우수모델.csv
  M23_부동산_임대_사업서비스업_Top10 우수모델.csv
  M25_오락_문화_개인서비스업_Top10 우수모델.csv

[업종] M03_음식료품_제조업
  [참고] Threshold 컬럼 없음 → 기본값 0.44 사용
  FeatureSet=top65_dedup47 | Threshold=0.44
  PD 데이터: 7837행 | 연도 2014~2024 | 기업 수 1203개
  저장 완료 → 중간결과\21_PD변화율\M03_음식료품_제조업_top65_dedup47_2015-2024_기업_부실확률_차이값.csv  (1203행)
  [위험 신호 분포]
    안정 : 580개
    데이터없음(최종연도) : 392개
    관찰 (최근가속) : 85개
    관찰 (최근2년상승) : 34개
    주의 (최근급등) : 26개
    관찰 (가속) : 21개
    위험 (최근가속+임계초과) : 19개
    관찰 (3년연속상승) : 10개
    위험 (최근급등+임계초과) : 8개
    위험 (가속+임계초과) : 8개
    주의 (최근가속+임계근접) : 6개
    보통 : 4개
    위험 (3년연속상승+임계초과) :

# PD

In [1]:
# ============================================================
# 0. 라이브러리
# ============================================================

import os
import re
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, average_precision_score, accuracy_score
)
from xgboost import XGBClassifier

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("[경고] LightGBM 미설치 → LightGBM 모델 사용 불가")

try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

try:
    from ctgan import CTGAN
    HAS_CTGAN = True
except ImportError:
    HAS_CTGAN = False

warnings.filterwarnings("ignore")


# ============================================================
# 1. 전역 경로 설정
# ============================================================
BASE_DIR         = r'중간결과'
TOP10_FOLDER     = os.path.join(BASE_DIR, r'15_우수모델\Top10')
TRAIN_FOLDER     = os.path.join(BASE_DIR, '12_train')
TEST_FOLDER      = os.path.join(BASE_DIR, '12_test')
FEATURE_FOLDER   = os.path.join(BASE_DIR, '13_피처셀렉션')

# 저장 폴더 분리
SAVE_PD_DIR      = os.path.join(BASE_DIR, r'15_우수모델\2014_2024_PD')
SAVE_SUMMARY_DIR = os.path.join(BASE_DIR, r'15_우수모델\요약')
SAVE_OTHERS_DIR  = os.path.join(BASE_DIR, r'15_우수모델\나머지')

for d in [SAVE_PD_DIR, SAVE_SUMMARY_DIR, SAVE_OTHERS_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42
YEAR_COL     = "회계년도"
ID_COLS      = ["회사명", "사업자등록번호", "회계년도"]
FOLD_VAL_YEARS = [2016, 2017, 2018, 2019, 2020, 2021]
TRAIN_START    = 2012
RECALL_MIN     = 0.9


# ============================================================
# 2. Top10 CSV → 업종코드/폴더명 추출 유틸
# ============================================================

def parse_industry_info(top10_csv_path: str):
    """
    파일명 예시: M03_음식료품_제조업_Top10 우수모델.csv
    → industry_code = "M03"
    → industry_folder = "M03_음식료품_제조업"   (13_피처셀렉션 하위 폴더명)
    → train_file = "M03_음식료품_제조업_train.parquet"
    → test_file  = "M03_음식료품_제조업_test.parquet"
    → prefix     = "M03_음식료품_제조업"
    """
    fname = os.path.basename(top10_csv_path)           # M03_음식료품_제조업_Top10 우수모델.csv
    # "_Top10" 이전 부분을 업종 식별자로 사용
    match = re.match(r'^(.+?)_Top10', fname)
    if not match:
        raise ValueError(f"파일명 패턴 불일치: {fname}")
    prefix = match.group(1)                             # M03_음식료품_제조업
    code   = prefix.split('_')[0]                       # M03

    return {
        "prefix"         : prefix,
        "code"           : code,
        "train_file"     : os.path.join(TRAIN_FOLDER, f"{prefix}_train.parquet"),
        "test_file"      : os.path.join(TEST_FOLDER,  f"{prefix}_test.parquet"),
        "feature_folder" : os.path.join(FEATURE_FOLDER, prefix),
    }


# ============================================================
# 3. 모델 팩토리 — Top10 Model 컬럼값 기반으로 모델 생성
# ============================================================

def make_model(model_name: str, pos_weight: float):
    """
    model_name : "XGBoost" / "LightGBM" / "RandomForest" / "LogisticRegression"
    pos_weight : ClassWeight 방식일 때 양성 클래스 가중치 (그 외 방식은 1.0)
    """
    mn = str(model_name).strip()

    if mn == "XGBoost":
        return XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="aucpr",
            random_state=RANDOM_STATE, verbosity=0,
            scale_pos_weight=pos_weight        # ClassWeight → pos_weight, 그 외 → 1.0
        )

    if mn == "LightGBM":
        if not HAS_LGBM:
            raise ImportError("LightGBM 미설치. pip install lightgbm")
        if pos_weight > 1.0:
            # ClassWeight 방식 → is_unbalance 대신 scale_pos_weight 사용
            return LGBMClassifier(
                n_estimators=300, learning_rate=0.05, max_depth=4,
                subsample=0.8, colsample_bytree=0.8,
                random_state=RANDOM_STATE, verbose=-1,
                scale_pos_weight=pos_weight
            )
        else:
            return LGBMClassifier(
                n_estimators=300, learning_rate=0.05, max_depth=4,
                subsample=0.8, colsample_bytree=0.8,
                random_state=RANDOM_STATE, verbose=-1
            )

    if mn == "RandomForest":
        if pos_weight > 1.0:
            return RandomForestClassifier(
                n_estimators=300, max_depth=4, min_samples_leaf=5,
                random_state=RANDOM_STATE, n_jobs=-1,
                class_weight="balanced"
            )
        else:
            return RandomForestClassifier(
                n_estimators=300, max_depth=4, min_samples_leaf=5,
                random_state=RANDOM_STATE, n_jobs=-1
            )

    if mn == "LogisticRegression":
        if pos_weight > 1.0:
            return LogisticRegression(
                penalty="l2", C=1.0, solver="lbfgs",
                max_iter=1000, random_state=RANDOM_STATE,
                class_weight="balanced"
            )
        else:
            return LogisticRegression(
                penalty="l2", C=1.0, solver="lbfgs",
                max_iter=1000, random_state=RANDOM_STATE
            )

    raise ValueError(f"지원하지 않는 모델명: {model_name}")


# ============================================================
# 4. 불균형 처리 함수
# ============================================================

def _parse_smote_ratio(smote_ratio):
    try:
        if smote_ratio is None:
            return None
        s = str(smote_ratio).strip()
        if s in ("", "-", "nan", "None"):
            return None
        return float(s)
    except (TypeError, ValueError):
        return None


def _ctgan_resample(X, y, ratio):
    """
    소수 클래스(y==1)를 CTGAN으로 학습해 합성 표본을 추가.

    ratio 해석 (SMOTE sampling_strategy와 동일 기준):
        ratio = 소수 목표 수 / 다수 수
        → 소수 목표 수 = int(n_majority * ratio)
        → n_synth = 소수 목표 수 - 현재 소수 수  (음수면 합성 불필요)

    ratio=None이면 1:1 균형(n_majority개)까지 합성.
    """
    if not HAS_CTGAN:
        raise ImportError("CTGAN 방식을 사용하려면 'pip install ctgan'이 필요합니다.")

    # ── DataFrame 보장 (컬럼명 보존)
    if not isinstance(X, pd.DataFrame):
        raise TypeError("_ctgan_resample: X는 pd.DataFrame이어야 합니다.")

    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)

    n_minority = int((y == 1).sum())
    n_majority = int((y == 0).sum())

    if n_minority < 5:
        print(f"    [CTGAN 경고] 소수 클래스 표본 수({n_minority})가 너무 적어 합성을 건너뜁니다.")
        return X, y

    # 목표 소수 클래스 수 계산
    if ratio is not None:
        target_minority = int(n_majority * ratio)   # 소수/다수 = ratio
    else:
        target_minority = n_majority                # 1:1 균형

    n_synth = target_minority - n_minority

    if n_synth <= 0:
        # 이미 목표 비율 이상 → 합성 불필요
        print(f"    [CTGAN] 현재 소수({n_minority}) >= 목표({target_minority}), 합성 불필요.")
        return X, y

    print(f"    [CTGAN] 소수={n_minority}, 다수={n_majority}, "
          f"ratio={ratio}, 목표소수={target_minority}, 합성수={n_synth}")

    minority_df = X[y == 1].copy().reset_index(drop=True)

    # verbose 파라미터: 버전에 따라 다를 수 있어 try/except로 처리
    try:
        ctgan = CTGAN(epochs=300, verbose=False)
    except TypeError:
        ctgan = CTGAN(epochs=300)

    ctgan.fit(minority_df)
    synth = ctgan.sample(n_synth)

    # 컬럼 순서/구성 맞춤
    synth = synth.reindex(columns=X.columns)

    X_res = pd.concat([X, synth], ignore_index=True)
    y_res = pd.concat([y, pd.Series([1] * n_synth)], ignore_index=True)

    print(f"    [CTGAN] 합성 완료 → 전체 {len(X_res)}행 "
          f"(정상={int((y_res==0).sum())}, 부실={int((y_res==1).sum())})")
    return X_res, y_res


def apply_resampling(X, y, method, smote_ratio):
    method_l = str(method).strip().lower()
    ratio    = _parse_smote_ratio(smote_ratio)

    if method_l == "classweight":
        pos_weight = (y == 0).sum() / (y == 1).sum()
        return X, y, pos_weight

    if method_l in ("none", "", "-", "nan"):
        return X, y, 1.0

    if "borderline" in method_l:
        if not HAS_IMBLEARN:
            raise ImportError("pip install imbalanced-learn")
        sampler = BorderlineSMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if method_l == "smote":
        if not HAS_IMBLEARN:
            raise ImportError("pip install imbalanced-learn")
        sampler = SMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if "ctgan" in method_l:
        X_res, y_res = _ctgan_resample(X, y, ratio)
        return X_res, y_res, 1.0

    print(f"  [경고] 알 수 없는 Method='{method}' → ClassWeight로 처리합니다.")
    pos_weight = (y == 0).sum() / (y == 1).sum()
    return X, y, pos_weight


# ============================================================
# 5. 평가 유틸
# ============================================================

def find_threshold_at_recall(y_true, y_prob, recall_min=RECALL_MIN):
    thresholds = np.arange(0.01, 1.0, 0.01)
    valid = []
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        rec = recall_score(y_true, y_pred, zero_division=0)
        if rec >= recall_min:
            valid.append(round(thr, 2))
    return max(valid) if valid else None


def calc_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "F1"        : f1_score(y_true, y_pred, zero_division=0),
        "Recall"    : recall_score(y_true, y_pred, zero_division=0),
        "Precision" : precision_score(y_true, y_pred, zero_division=0),
        "ROC_AUC"   : roc_auc_score(y_true, y_prob),
        "PR_AUC"    : average_precision_score(y_true, y_prob),
        "Accuracy"  : accuracy_score(y_true, y_pred),
    }


# ============================================================
# 6. 시각화 함수
# ============================================================

plt.rcParams.update({
    "font.family"       : "DejaVu Sans",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : True,
    "grid.color"        : "#E5E5E5",
    "grid.linewidth"    : 0.7,
    "axes.facecolor"    : "#FAFAFA",
    "figure.facecolor"  : "white",
})

COLOR_NEG  = "#2F6EBA"
COLOR_POS  = "#D94F3D"
COLOR_THR  = "#F5A623"
ALPHA_HIST = 0.72
BINS       = 45


def plot_dist(ax, y_true, y_prob, title, n_total, threshold):
    arr0 = y_prob[np.array(y_true) == 0]
    arr1 = y_prob[np.array(y_true) == 1]
    counts0, edges0 = np.histogram(arr0, bins=BINS, range=(0, 1))
    counts1, edges1 = np.histogram(arr1, bins=BINS, range=(0, 1))
    ax.bar(edges0[:-1], counts0, width=np.diff(edges0),
           align="edge", color=COLOR_NEG, alpha=ALPHA_HIST, label="Normal (0)", zorder=3)
    ax.bar(edges1[:-1], counts1, width=np.diff(edges1),
           align="edge", color=COLOR_POS, alpha=ALPHA_HIST, label="Distress (1)", zorder=3)
    ax.axvline(threshold, color=COLOR_THR, linestyle="--",
               linewidth=1.8, zorder=5, label=f"Threshold = {threshold:.2f}")
    ax.axvspan(threshold, 1.0, alpha=0.06, color=COLOR_POS, zorder=2)
    n0, n1 = len(arr0), len(arr1)
    ir = n1 / n0 if n0 > 0 else float("nan")
    above_thr = (y_prob >= threshold).sum()
    stats_txt = (
        f"N={n_total:,}  |  Normal={n0:,}  Distress={n1:,}\n"
        f"Imbalance ratio = {ir:.3f}  |  Predicted Positive = {above_thr:,}"
    )
    ax.text(0.98, 0.97, stats_txt, transform=ax.transAxes, fontsize=8.2,
            va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#CCCCCC", alpha=0.85))
    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    ax.set_xlabel("Predicted Probability", fontsize=9.5)
    ax.set_ylabel("Count", fontsize=9.5)
    ax.set_xlim(0, 1)
    ax.tick_params(labelsize=8.5)
    legend_elems = [
        Patch(facecolor=COLOR_NEG, alpha=ALPHA_HIST, label="Normal (0)"),
        Patch(facecolor=COLOR_POS, alpha=ALPHA_HIST, label="Distress (1)"),
        Line2D([0], [0], color=COLOR_THR, linestyle="--", linewidth=1.8,
               label=f"Threshold = {threshold:.2f}"),
    ]
    ax.legend(handles=legend_elems, fontsize=8.5, framealpha=0.9,
              loc="upper left", edgecolor="#CCCCCC")


def save_prob_dist_plot(val_prob_df, y_test, y_prob_test, threshold,
                        method, feature_set, model_name, prefix):
    fig = plt.figure(figsize=(16, 10))
    gs  = gridspec.GridSpec(
        2, 2, height_ratios=[3.2, 1],
        hspace=0.42, wspace=0.32,
        left=0.07, right=0.97, top=0.91, bottom=0.05
    )
    ax_cv   = fig.add_subplot(gs[0, 0])
    ax_test = fig.add_subplot(gs[0, 1])
    ax_tbl  = fig.add_subplot(gs[1, :])
    ax_tbl.axis("off")

    plot_dist(ax_cv, val_prob_df["y_true"].values, val_prob_df["y_prob"].values,
              "Expanding Window CV — Predicted Probability Distribution",
              n_total=len(val_prob_df), threshold=threshold)
    plot_dist(ax_test, y_test.values, y_prob_test,
              "Hold-out Test — Predicted Probability Distribution",
              n_total=len(y_test), threshold=threshold)

    # 테이블
    cv_mean_      = val_prob_df.groupby("Val_Year").apply(
        lambda g: pd.Series(calc_metrics(g["y_true"], g["y_prob"], threshold))
    ).mean()
    test_metrics_ = calc_metrics(y_test, y_prob_test, threshold)
    metrics_order = ["F1", "Recall", "Precision", "ROC_AUC", "PR_AUC", "Accuracy"]
    col_labels    = ["Split"] + metrics_order
    cv_row   = ["CV Mean"] + [f"{float(cv_mean_[m]):.4f}" for m in metrics_order]
    test_row = ["Test"]    + [f"{test_metrics_[m]:.4f}"   for m in metrics_order]
    gap_row  = ["Gap (Test − CV)"] + [
        f"{test_metrics_[m] - float(cv_mean_[m]):+.4f}" for m in metrics_order
    ]
    table_data = [cv_row, test_row, gap_row]
    tbl = ax_tbl.table(cellText=table_data, colLabels=col_labels,
                        cellLoc="center", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.7)
    for j in range(len(col_labels)):
        tbl[(0, j)].set_facecolor("#2F4F7F")
        tbl[(0, j)].set_text_props(color="white", fontweight="bold")
    row_colors = ["#EEF3FA", "#FAFAFA", "#FFF4EE"]
    for i, rc in enumerate(row_colors, start=1):
        for j in range(len(col_labels)):
            tbl[(i, j)].set_facecolor(rc)
    for j, m in enumerate(metrics_order, start=1):
        gap_val = test_metrics_[m] - float(cv_mean_[m])
        color   = "#C0392B" if gap_val < -0.02 else ("#27AE60" if gap_val > 0.02 else "#555555")
        tbl[(3, j)].set_text_props(color=color, fontweight="bold")
    ax_tbl.set_title("Performance Summary", fontsize=10, fontweight="bold", pad=6, loc="left")

    fig.suptitle(
        f"{model_name} | {method} | {feature_set} | {prefix} | "
        f"Threshold(Recall≥{RECALL_MIN}) = {threshold:.2f}",
        fontsize=13.5, fontweight="bold", y=0.975
    )
    save_path = os.path.join(SAVE_OTHERS_DIR, f"{prefix}_{feature_set}_prob_distribution.png")
    plt.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.close()
    return save_path


# ============================================================
# 7. 단일 업종 처리 함수
# ============================================================

def process_one_industry(top10_csv_path: str):
    info   = parse_industry_info(top10_csv_path)
    prefix = info["prefix"]

    print(f"\n{'='*70}")
    print(f"[업종] {prefix}")
    print(f"{'='*70}")

    # ── Top10 1위 행 로드
    top10 = pd.read_csv(top10_csv_path, index_col=0)
    if len(top10) == 0:
        print(f"  [경고] {top10_csv_path} 가 비어있습니다. skip.")
        return

    row          = top10.iloc[0]
    FEATURE_SET  = row["FeatureSet"]
    FEATURE_FILE = row["FeatureFile"]
    METHOD       = row["Method"]
    SMOTE_RATIO  = row["SMOTE_Ratio"]
    MODEL_NAME   = row["Model"]

    print(f"  FeatureSet  : {FEATURE_SET}")
    print(f"  FeatureFile : {FEATURE_FILE}")
    print(f"  Method      : {METHOD}")
    print(f"  SMOTE_Ratio : {SMOTE_RATIO}")
    print(f"  Model       : {MODEL_NAME}")

    # ── 피처 파일 경로 구성
    # FEATURE_FILE 예: "lasso_features_top50--41.csv"
    # 경로 예: 중간결과\13_피처셀렉션\M03_음식료품_제조업\lasso_features_top50--41.csv
    feature_path = os.path.join(info["feature_folder"], FEATURE_FILE)
    if not os.path.exists(feature_path):
        print(f"  [경고] 피처 파일 없음: {feature_path} → skip")
        return

    # ── 데이터 로드
    if not os.path.exists(info["train_file"]):
        print(f"  [경고] train 파일 없음: {info['train_file']} → skip")
        return
    if not os.path.exists(info["test_file"]):
        print(f"  [경고] test 파일 없음: {info['test_file']} → skip")
        return

    train_full = pd.read_parquet(info["train_file"])
    test       = pd.read_parquet(info["test_file"])
    y_train_full = train_full[TARGET_COL]
    y_test       = test[TARGET_COL]

    feat_df      = pd.read_csv(feature_path)
    col_key      = "feature" if "feature" in feat_df.columns else feat_df.columns[0]
    use_features = [f for f in feat_df[col_key].tolist() if f in train_full.columns]

    print(f"\n  Train shape : {train_full.shape} | Test shape : {test.shape}")
    print(f"  피처 수      : {len(use_features)}개  | 양성비율(train): "
          f"{y_train_full.mean()*100:.2f}%")

    # ── Expanding Window CV
    print(f"\n  ── Expanding Window CV")
    fold_results = []
    for val_year in FOLD_VAL_YEARS:
        train_idx = train_full.index[
            (train_full[YEAR_COL] >= TRAIN_START) & (train_full[YEAR_COL] < val_year)
        ]
        val_idx = train_full.index[train_full[YEAR_COL] == val_year]
        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        X_ft = train_full.loc[train_idx, use_features]
        y_ft = train_full.loc[train_idx, TARGET_COL]
        X_fv = train_full.loc[val_idx,   use_features]
        y_fv = train_full.loc[val_idx,   TARGET_COL]

        imp = SimpleImputer(strategy="median")
        X_ft = pd.DataFrame(imp.fit_transform(X_ft), columns=use_features)
        X_fv = pd.DataFrame(imp.transform(X_fv),     columns=use_features)

        X_res, y_res, pw = apply_resampling(X_ft, y_ft, METHOD, SMOTE_RATIO)
        model = make_model(MODEL_NAME, pw)
        model.fit(X_res, y_res)
        y_prob_val = model.predict_proba(X_fv)[:, 1]

        print(f"    Val {val_year} | ROC_AUC={roc_auc_score(y_fv, y_prob_val):.4f} "
              f"PR_AUC={average_precision_score(y_fv, y_prob_val):.4f}")
        fold_results.append({
            "Val_Year": val_year,
            "y_true"  : y_fv.values,
            "y_prob"  : y_prob_val
        })

    # ── 최종 모델 학습 (전체 train)
    imputer_final = SimpleImputer(strategy="median")
    X_train_all   = pd.DataFrame(
        imputer_final.fit_transform(train_full[use_features]), columns=use_features
    )
    X_test_imp    = pd.DataFrame(
        imputer_final.transform(test[use_features]), columns=use_features
    )
    X_train_res, y_train_res, final_pw = apply_resampling(
        X_train_all, y_train_full, METHOD, SMOTE_RATIO
    )
    final_model = make_model(MODEL_NAME, final_pw)
    final_model.fit(X_train_res, y_train_res)
    y_prob_test = final_model.predict_proba(X_test_imp)[:, 1]

    # ── Threshold 탐색 (Test 기준, Recall >= RECALL_MIN)
    found_thr = find_threshold_at_recall(y_test, y_prob_test, RECALL_MIN)
    THRESHOLD = found_thr if found_thr is not None else 0.01
    if found_thr is None:
        print(f"  [경고] Recall>={RECALL_MIN} 만족하는 threshold 없음 → 0.01 사용")

    test_metrics = calc_metrics(y_test, y_prob_test, THRESHOLD)
    print(f"\n  Test threshold = {THRESHOLD:.2f} | "
          f"Recall={test_metrics['Recall']:.4f} "
          f"Precision={test_metrics['Precision']:.4f} "
          f"F1={test_metrics['F1']:.4f}")

    # ── CV fold별 성능 (Test threshold 적용)
    cv_rows      = []
    val_prob_all = []
    for fr in fold_results:
        m = calc_metrics(fr["y_true"], fr["y_prob"], THRESHOLD)
        cv_rows.append({"Val_Year": fr["Val_Year"], **m})
        val_prob_all.append(pd.DataFrame({
            "Val_Year": fr["Val_Year"],
            "y_true"  : fr["y_true"],
            "y_prob"  : fr["y_prob"],
            "y_pred"  : (fr["y_prob"] >= THRESHOLD).astype(int),
        }))

    cv_df       = pd.DataFrame(cv_rows).round(4)
    cv_mean     = cv_df[["F1","Recall","Precision","ROC_AUC","PR_AUC","Accuracy"]].mean()
    val_prob_df = pd.concat(val_prob_all, ignore_index=True) if val_prob_all else pd.DataFrame()

    # ── 파일명 공통 접두사
    file_prefix = f"{prefix}_{FEATURE_SET}"

    # ────────────────────────────────────────────────
    # 저장 ①: 요약 (summary)
    # ────────────────────────────────────────────────
    summary = {
        "Industry"         : prefix,
        "FeatureSet"       : FEATURE_SET,
        "FeatureFile"      : FEATURE_FILE,
        "N_Features"       : len(use_features),
        "Method"           : METHOD,
        "SMOTE_Ratio"      : SMOTE_RATIO,
        "Model"            : MODEL_NAME,
        "Recall_min"       : RECALL_MIN,
        "Threshold_basis"  : "Test",
        "Threshold"        : THRESHOLD,
    }
    for col in ["F1","Recall","Precision","ROC_AUC","PR_AUC","Accuracy"]:
        summary[f"CV_Val_{col}"] = round(float(cv_mean[col]), 4)
        summary[f"Test_{col}"]   = round(test_metrics[col], 4)
        summary[f"Gap_{col}"]    = round(test_metrics[col] - float(cv_mean[col]), 4)

    pd.DataFrame([summary]).to_csv(
        os.path.join(SAVE_SUMMARY_DIR, f"{file_prefix}_summary.csv"),
        index=False, encoding="utf-8-sig"
    )

    # ────────────────────────────────────────────────
    # 저장 ②: 나머지 (CV 결과, 확률분포 이미지, test predictions)
    # ────────────────────────────────────────────────
    cv_df.to_csv(
        os.path.join(SAVE_OTHERS_DIR, f"{file_prefix}_CV_results.csv"),
        index=False, encoding="utf-8-sig"
    )

    y_pred_test = (y_prob_test >= THRESHOLD).astype(int)
    test_id_cols = [c for c in ID_COLS if c in test.columns]
    test_pred_df = test[test_id_cols].copy().reset_index(drop=True)
    test_pred_df["y_true"]  = y_test.values
    test_pred_df["y_prob"]  = y_prob_test.round(4)
    test_pred_df["y_pred"]  = y_pred_test
    test_pred_df["correct"] = (test_pred_df["y_true"] == test_pred_df["y_pred"]).astype(int)
    test_pred_df["error_type"] = "TN"
    test_pred_df.loc[(test_pred_df["y_true"]==1)&(test_pred_df["y_pred"]==1), "error_type"] = "TP"
    test_pred_df.loc[(test_pred_df["y_true"]==1)&(test_pred_df["y_pred"]==0), "error_type"] = "FN"
    test_pred_df.loc[(test_pred_df["y_true"]==0)&(test_pred_df["y_pred"]==1), "error_type"] = "FP"
    test_pred_df.to_csv(
        os.path.join(SAVE_OTHERS_DIR, f"{file_prefix}_test_predictions.csv"),
        index=False, encoding="utf-8-sig"
    )

    if not val_prob_df.empty:
        img_path = save_prob_dist_plot(
            val_prob_df, y_test, y_prob_test, THRESHOLD,
            METHOD, FEATURE_SET, MODEL_NAME, prefix
        )
        print(f"  확률분포 이미지 → {img_path}")

    # ────────────────────────────────────────────────
    # 저장 ③: 2014~2024 PD 데이터
    # ────────────────────────────────────────────────
    PD_YEARS = list(range(2014, 2025))
    all_data = pd.concat([train_full, test], ignore_index=True)
    ady      = all_data[all_data[YEAR_COL].isin(PD_YEARS)].copy()

    X_ady_imp = pd.DataFrame(
        imputer_final.transform(ady[use_features]),
        columns=use_features, index=ady.index
    )
    ady["y_prob"]  = final_model.predict_proba(X_ady_imp)[:, 1].round(4)
    ady["y_pred"]  = (ady["y_prob"] >= THRESHOLD).astype(int)
    ady["y_true"]  = ady[TARGET_COL]
    ady["correct"] = (ady["y_true"] == ady["y_pred"]).astype(int)
    ady["error_type"] = "TN"
    ady.loc[(ady["y_true"]==1)&(ady["y_pred"]==1), "error_type"] = "TP"
    ady.loc[(ady["y_true"]==1)&(ady["y_pred"]==0), "error_type"] = "FN"
    ady.loc[(ady["y_true"]==0)&(ady["y_pred"]==1), "error_type"] = "FP"

    id_cols_avail = [c for c in ID_COLS if c in ady.columns]
    pd_df = ady[id_cols_avail + ["y_true","y_prob","y_pred","correct","error_type"]] \
        .sort_values([id_cols_avail[1], YEAR_COL]).reset_index(drop=True)

    pd_save_path = os.path.join(SAVE_PD_DIR, f"{file_prefix}_2014_2024_PD_데이터.csv")
    pd_df.to_csv(pd_save_path, index=False, encoding="utf-8-sig")
    print(f"  PD 데이터(2014~2024) → {pd_save_path}  ({len(pd_df)}행)")
    print(f"  error_type 분포:\n{pd_df['error_type'].value_counts().to_string()}")

    return summary


# ============================================================
# 8. 전체 Top10 파일 순회 실행
# ============================================================

def main():
    top10_files = sorted(glob.glob(os.path.join(TOP10_FOLDER, "*.csv")))
    if not top10_files:
        print(f"[오류] Top10 파일이 없습니다: {TOP10_FOLDER}")
        return

    # ── 스킵할 업종 파일명 (부분 문자열 매칭)
    SKIP_KEYWORDS = [
        "M03_음식료품_제조업",
        "M04_섬유_가죽_신발_제조업",
        "M08_화학_의약품_고무_플라스틱_제조업"
    ]

    skip_files  = [f for f in top10_files if any(kw in os.path.basename(f) for kw in SKIP_KEYWORDS)]
    run_files   = [f for f in top10_files if f not in skip_files]

    print(f"전체 업종 수  : {len(top10_files)}개")
    print(f"스킵 업종 수  : {len(skip_files)}개")
    for f in skip_files:
        print(f"  [SKIP] {os.path.basename(f)}")
    print(f"처리할 업종 수: {len(run_files)}개")
    for f in run_files:
        print(f"  {os.path.basename(f)}")

    all_summaries = []
    failed = []

    for top10_csv in run_files:
        try:
            result = process_one_industry(top10_csv)
            if result:
                all_summaries.append(result)
        except Exception as e:
            industry_name = os.path.basename(top10_csv)
            print(f"\n  [오류] {industry_name} 처리 중 예외 발생: {e}")
            failed.append({"file": industry_name, "error": str(e)})

    # ── 전체 요약 통합 CSV
    if all_summaries:
        all_summary_df = pd.DataFrame(all_summaries)
        all_summary_df.to_csv(
            os.path.join(SAVE_SUMMARY_DIR, "전체_업종_통합_summary.csv"),
            index=False, encoding="utf-8-sig"
        )
        print(f"\n\n{'='*70}")
        print(f"전체 처리 완료: 성공 {len(all_summaries)}개 / 실패 {len(failed)}개")
        print(f"통합 요약 → {os.path.join(SAVE_SUMMARY_DIR, '전체_업종_통합_summary.csv')}")
        print(f"{'='*70}")

    if failed:
        print(f"\n[실패 목록]")
        for f in failed:
            print(f"  {f['file']} : {f['error']}")
        pd.DataFrame(failed).to_csv(
            os.path.join(SAVE_OTHERS_DIR, "실패_목록.csv"),
            index=False, encoding="utf-8-sig"
        )


if __name__ == "__main__":
    main()

전체 업종 수  : 15개
스킵 업종 수  : 3개
  [SKIP] M03_음식료품_제조업_Top10 우수모델.csv
  [SKIP] M04_섬유_가죽_신발_제조업_Top10 우수모델.csv
  [SKIP] M08_화학_의약품_고무_플라스틱_제조업_Top10 우수모델.csv
처리할 업종 수: 12개
  M10_제1차금속산업_Top10 우수모델.csv
  M11_조립금속제품_제조업_Top10 우수모델.csv
  M12_기타기계장비_제조업_Top10 우수모델.csv
  M13_전자부품_컴퓨터_전기장비_제조업_Top10 우수모델.csv
  M15_운송장비_제조업_Top10 우수모델.csv
  M17_전기_가스_수도사업_Top10 우수모델.csv
  M18_건설업_Top10 우수모델.csv
  M20_숙박_음식점업_Top10 우수모델.csv
  M21_운수_창고업_Top10 우수모델.csv
  M22_정보통신업_Top10 우수모델.csv
  M23_부동산_임대_사업서비스업_Top10 우수모델.csv
  M25_오락_문화_개인서비스업_Top10 우수모델.csv

[업종] M10_제1차금속산업
  FeatureSet  : top65_dedup44
  FeatureFile : lasso_features_top65--44.csv
  Method      : CTGAN
  SMOTE_Ratio : 0.2
  Model       : RandomForest

  Train shape : (6531, 256) | Test shape : (1954, 256)
  피처 수      : 44개  | 양성비율(train): 4.43%

  ── Expanding Window CV
    [CTGAN] 소수=149, 다수=2526, ratio=0.2, 목표소수=505, 합성수=356
    [CTGAN] 합성 완료 → 전체 3031행 (정상=2526, 부실=505)
    Val 2016 | ROC_AUC=0.8193 PR_AUC=0.1265
    [CTGAN] 소수=167, 다수=31

# score

### 통합데이터

In [11]:
# -*- coding: utf-8 -*-
# ============================================================
# 전체 업종 EWS 종합 위험 스코어 통합 스크립트
#
# 입력 경로:
#   중간결과\17_기업충격민감도\{prefix}\충격민감도_OLS_{prefix}.csv
#   중간결과\18_산업충격민감도\{prefix}\산업충격민감도_OLS.csv
#   중간결과\19_포터5F\{prefix}\마이클포터5F.csv
#   중간결과\20_생애주기\{prefix}\{prefix}_lifecycle_scored_yearly_minmax.csv
#   중간결과\21_PD변화율\{prefix}_{FEATURE_SET}_2015-2024_기업_부실확률_차이값.csv
#
# 출력 경로:
#   중간결과\22_SCORE\{prefix}_통합_스코어_데이터.csv
# ============================================================

import os
import re
import glob
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# ============================================================
# 1. 전역 경로 설정
# ============================================================
BASE_DIR      = r'중간결과'
TOP10_FOLDER  = os.path.join(BASE_DIR, r'15_우수모델\Top10')
DIR_FIRM_BETA = os.path.join(BASE_DIR, '17_기업충격민감도')
DIR_IND_BETA  = os.path.join(BASE_DIR, '18_산업충격민감도')
DIR_PORTER    = os.path.join(BASE_DIR, '19_포터5F')
DIR_LIFECYCLE = os.path.join(BASE_DIR, '20_생애주기')
DIR_PD        = os.path.join(BASE_DIR, '21_PD변화율')
SAVE_DIR      = os.path.join(BASE_DIR, '22_SCORE')
os.makedirs(SAVE_DIR, exist_ok=True)

# True: 10개 연도 전체 통합 min-max / False: 연도별 횡단면 min-max
GLOBAL_MINMAX = False


# ============================================================
# 2. 유틸 함수
# ============================================================

def read_csv_safe(path, **kwargs):
    """사업자등록번호를 문자열(10자리 zfill)로 안전하게 읽는 CSV 로더"""
    df = pd.read_csv(path, dtype={"사업자등록번호": str}, **kwargs)
    if "사업자등록번호" in df.columns:
        df["사업자등록번호"] = df["사업자등록번호"].str.zfill(10)
    return df


def minmax(s: pd.Series) -> pd.Series:
    """NaN 무시 0~100 min-max. max==min이면 50 처리."""
    mn, mx = s.min(skipna=True), s.max(skipna=True)
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(np.where(s.notna(), 50.0, np.nan), index=s.index)
    return (s - mn) / (mx - mn) * 100


def minmax_by_year(df, value_col, year_col="연도"):
    """연도별(횡단면) 0~100 min-max"""
    return df.groupby(year_col)[value_col].transform(minmax)


def parse_prefix(top10_csv_path: str) -> str:
    fname = os.path.basename(top10_csv_path)
    match = re.match(r'^(.+?)_Top10', fname)
    if not match:
        raise ValueError(f"파일명 패턴 불일치: {fname}")
    return match.group(1)


# ============================================================
# 3. 입력 파일 경로 구성 함수
# ============================================================

def get_paths(prefix: str, feature_set: str) -> dict:
    return {
        # lifecycle 파일명 패턴 두 가지 대응:
        # 1) {prefix}_lifecycle_scored_yearly_minmax.csv  (M03~M15 계열)
        # 2) lifecycle_scored_yearly_minmax.csv           (M17~ 계열, prefix 없음)
        "lifecycle" : os.path.join(DIR_LIFECYCLE, prefix,
            f"{prefix}_lifecycle_scored_yearly_minmax.csv"
        ),
        "lifecycle_short" : os.path.join(DIR_LIFECYCLE, prefix,
            "lifecycle_scored_yearly_minmax.csv"
        ),
        "porter"    : os.path.join(DIR_PORTER, prefix, "마이클포터5F.csv"),
        "ind_beta"  : os.path.join(DIR_IND_BETA, prefix, "산업충격민감도_OLS.csv"),
        "firm_beta" : os.path.join(
            DIR_FIRM_BETA, prefix,
            f"충격민감도_OLS_{prefix}.csv"
        ),
        "pd_change" : os.path.join(
            DIR_PD,
            f"{prefix}_{feature_set}_2015-2024_기업_부실확률_차이값.csv"
        ),
    }


# ============================================================
# 4. 단일 업종 처리 함수
# ============================================================

def process_one_industry(top10_csv_path: str):
    prefix = parse_prefix(top10_csv_path)

    print(f"\n{'='*70}")
    print(f"[업종] {prefix}")
    print(f"{'='*70}")

    # ── Top10 1위 행에서 FEATURE_SET 가져오기
    top10 = pd.read_csv(top10_csv_path, index_col=0)
    if len(top10) == 0:
        print(f"  [경고] Top10 파일이 비어있습니다. skip.")
        return None

    FEATURE_SET = top10.iloc[0]["FeatureSet"]
    paths = get_paths(prefix, FEATURE_SET)

    # ── 파일 존재 확인
    # lifecycle은 두 패턴 중 하나만 있으면 OK → 둘 다 없을 때만 필수 누락
    lifecycle_ok = os.path.exists(paths["lifecycle"]) or os.path.exists(paths["lifecycle_short"])
    pd_ok        = os.path.exists(paths["pd_change"])

    for k, v in paths.items():
        if k in ("lifecycle", "lifecycle_short"):
            continue  # lifecycle은 아래에서 별도 처리
        if not os.path.exists(v):
            print(f"  [경고] {k} 파일 없음: {v}")

    if not lifecycle_ok:
        print(f"  [경고] lifecycle 파일 없음 (두 패턴 모두 미존재) → skip")
        return None
    if not pd_ok:
        print(f"  [경고] pd_change 파일 없음: {paths['pd_change']} → skip")
        return None

    # ──────────────────────────────────────────────────────────
    # Step 1. 베이스 테이블: lifecycle
    # ──────────────────────────────────────────────────────────
    # lifecycle 파일명 두 패턴 순서대로 시도
    # 1순위: {prefix}_lifecycle_scored_yearly_minmax.csv
    # 2순위: lifecycle_scored_yearly_minmax.csv (prefix 없음)
    lifecycle_path = None
    for pk in ["lifecycle", "lifecycle_short"]:
        if os.path.exists(paths[pk]):
            lifecycle_path = paths[pk]
            print(f"  lifecycle 파일: {os.path.basename(lifecycle_path)}")
            break

    if lifecycle_path is None:
        print(f"  [경고] lifecycle 파일 없음 → skip")
        return None

    lifecycle = read_csv_safe(lifecycle_path)

    # 기준연도 또는 연도 컬럼 대응
    if "기준연도" in lifecycle.columns:
        lifecycle = lifecycle.rename(columns={"기준연도": "연도"})

    base_cols = ["사업자등록번호", "연도", "생애주기_최종", "생애주기_점수", "부실라벨_ICR3년"]
    if "회계년도" in lifecycle.columns:
        base_cols = ["사업자등록번호", "연도", "회계년도", "생애주기_최종", "생애주기_점수", "부실라벨_ICR3년"]
    base = lifecycle[[c for c in base_cols if c in lifecycle.columns]].copy()

    # ──────────────────────────────────────────────────────────
    # Step 2. 포터 5F (산업 단위)
    # ──────────────────────────────────────────────────────────
    porter_df = None
    if os.path.exists(paths["porter"]):
        porter_raw = read_csv_safe(paths["porter"])
        if "연도" in porter_raw.columns and "최종점수" in porter_raw.columns:
            porter_raw = porter_raw[["연도", "최종점수"]].copy()
            porter_raw["Porter5F"] = minmax(porter_raw["최종점수"])
            porter_df = porter_raw[["연도", "Porter5F"]]
        else:
            print(f"  [경고] 포터5F 컬럼 불일치: {porter_raw.columns.tolist()}")
    else:
        print(f"  [경고] 포터5F 파일 없음 → NaN 처리")

    # ──────────────────────────────────────────────────────────
    # Step 3. 산업충격민감도 (산업 단위)
    # ──────────────────────────────────────────────────────────
    ind_beta_df = None
    if os.path.exists(paths["ind_beta"]):
        ind_raw = read_csv_safe(paths["ind_beta"])
        year_col = "테스트_연도" if "테스트_연도" in ind_raw.columns else "연도"
        if year_col in ind_raw.columns and "beta_i" in ind_raw.columns:
            ind_raw = ind_raw.rename(columns={year_col: "연도"})[["연도", "beta_i"]].copy()
            ind_raw["산업충격민감도"] = minmax(ind_raw["beta_i"])
            ind_beta_df = ind_raw[["연도", "산업충격민감도"]]
        else:
            print(f"  [경고] 산업충격민감도 컬럼 불일치: {ind_raw.columns.tolist()}")
    else:
        print(f"  [경고] 산업충격민감도 파일 없음 → NaN 처리")

    # ──────────────────────────────────────────────────────────
    # Step 4. 기업충격민감도 (기업 단위)
    # ──────────────────────────────────────────────────────────
    firm_beta_df = None
    firm_name_map = None
    if os.path.exists(paths["firm_beta"]):
        firm_raw = read_csv_safe(paths["firm_beta"])
        year_col = "테스트_연도" if "테스트_연도" in firm_raw.columns else "연도"
        if year_col in firm_raw.columns and "beta_i" in firm_raw.columns:
            firm_raw = firm_raw.rename(columns={year_col: "연도"})
            cols = ["사업자등록번호", "연도", "beta_i"]
            if "회사명" in firm_raw.columns:
                cols = ["사업자등록번호", "회사명", "연도", "beta_i"]
            firm_raw = firm_raw[cols].copy()

            if GLOBAL_MINMAX:
                firm_raw["기업충격민감도"] = minmax(firm_raw["beta_i"])
            else:
                firm_raw["기업충격민감도"] = minmax_by_year(firm_raw, "beta_i")

            firm_beta_df = firm_raw[["사업자등록번호", "연도", "기업충격민감도"]]
            if "회사명" in firm_raw.columns:
                firm_name_map = (
                    firm_raw[["사업자등록번호", "회사명"]]
                    .drop_duplicates(subset=["사업자등록번호"])
                )
        else:
            print(f"  [경고] 기업충격민감도 컬럼 불일치: {firm_raw.columns.tolist()}")
    else:
        print(f"  [경고] 기업충격민감도 파일 없음 → NaN 처리")

    # ──────────────────────────────────────────────────────────
    # Step 5. 부실확률 차이값 (기업 단위, wide → long)
    # ──────────────────────────────────────────────────────────
    prob = read_csv_safe(paths["pd_change"])
    years = range(2015, 2025)
    long_rows = []
    for y in years:
        prob_col = f"prob_{y}"
        diff_col = f"diff_{y-1}_{y}"
        if prob_col not in prob.columns or diff_col not in prob.columns:
            continue
        tmp = prob[["사업자등록번호", prob_col, diff_col]].copy()
        if "회사명" in prob.columns:
            tmp["회사명"] = prob["회사명"]
        tmp = tmp.rename(columns={prob_col: "prob", diff_col: "diff"})
        tmp["연도"] = y
        long_rows.append(tmp)

    prob_long = pd.concat(long_rows, ignore_index=True)

    if GLOBAL_MINMAX:
        prob_long["부실확률"] = minmax(prob_long["prob"])
        prob_long["부실확률변화"] = minmax(prob_long["diff"])
    else:
        prob_long["부실확률"] = minmax_by_year(prob_long, "prob")
        prob_long["부실확률변화"] = minmax_by_year(prob_long, "diff")

    prob_cols = ["사업자등록번호", "연도", "부실확률", "부실확률변화"]
    if "회사명" in prob_long.columns:
        prob_cols = ["사업자등록번호", "회사명", "연도", "부실확률", "부실확률변화"]
    prob_for_merge = prob_long[prob_cols]

    # ──────────────────────────────────────────────────────────
    # Step 6. 전체 병합
    # ──────────────────────────────────────────────────────────
    df = base.copy()

    if porter_df is not None:
        df = df.merge(porter_df, on="연도", how="left")
    else:
        df["Porter5F"] = np.nan

    if ind_beta_df is not None:
        df = df.merge(ind_beta_df, on="연도", how="left")
    else:
        df["산업충격민감도"] = np.nan

    if firm_beta_df is not None:
        df = df.merge(firm_beta_df, on=["사업자등록번호", "연도"], how="left")
    else:
        df["기업충격민감도"] = np.nan

    df = df.merge(prob_for_merge, on=["사업자등록번호", "연도"], how="left")

    # 회사명 채우기
    if "회사명" not in df.columns:
        df["회사명"] = np.nan
    if firm_name_map is not None:
        df["회사명"] = df["회사명"].combine_first(
            df["사업자등록번호"].map(
                firm_name_map.set_index("사업자등록번호")["회사명"]
            )
        )

    # ──────────────────────────────────────────────────────────
    # Step 7. 최종 컬럼 정리 + 생애주기_점수 → 생애주기점수
    # ──────────────────────────────────────────────────────────
    df = df.rename(columns={"생애주기_점수": "생애주기점수"})

    final_cols = [
        "사업자등록번호", "회사명", "연도",
        "생애주기_최종", "부실라벨_ICR3년",
        "Porter5F",
        "생애주기점수",
        "산업충격민감도",
        "기업충격민감도",
        "부실확률",
        "부실확률변화",
    ]
    # 없는 컬럼은 NaN으로 채움
    for c in final_cols:
        if c not in df.columns:
            df[c] = np.nan

    df_final = (
        df[final_cols]
        .sort_values(["사업자등록번호", "연도"])
        .reset_index(drop=True)
    )

    score_cols = [
        "생애주기점수", "Porter5F", "산업충격민감도",
        "기업충격민감도", "부실확률", "부실확률변화",
    ]
    df_final[score_cols] = df_final[score_cols].round(2)

    # 부실확률 결측 행 삭제
    before_n = len(df_final)
    df_final = df_final.dropna(subset=["부실확률"]).reset_index(drop=True)
    after_n  = len(df_final)
    print(f"  부실확률 결측 삭제: {before_n} → {after_n}행 ({before_n-after_n}행 제거)")

    # ──────────────────────────────────────────────────────────
    # Step 8. 저장
    # ──────────────────────────────────────────────────────────
    save_path = os.path.join(SAVE_DIR, f"{prefix}_통합_스코어_데이터.csv")
    df_final.to_csv(save_path, index=False, encoding="utf-8-sig")

    print(f"  저장 완료 → {save_path}  ({df_final.shape[0]}행)")
    print(f"  [결측치 비율]")
    print("  " + df_final[score_cols].isna().mean().round(3).to_string().replace("\n", "\n  "))
    print(f"  [점수 범위 확인 (min/max)]")
    print("  " + df_final[score_cols].agg(["min","max"]).round(2).to_string().replace("\n", "\n  "))

    return {
        "Industry"   : prefix,
        "FeatureSet" : FEATURE_SET,
        "N_Rows"     : len(df_final),
        "SavePath"   : save_path,
    }


# ============================================================
# 5. 전체 Top10 파일 순회 실행
# ============================================================

def main():
    top10_files = sorted(glob.glob(os.path.join(TOP10_FOLDER, "*.csv")))
    if not top10_files:
        print(f"[오류] Top10 파일이 없습니다: {TOP10_FOLDER}")
        return

    print(f"전체 업종 수: {len(top10_files)}개")
    for f in top10_files:
        print(f"  {os.path.basename(f)}")

    results = []
    failed  = []

    for top10_csv in top10_files:
        try:
            res = process_one_industry(top10_csv)
            if res:
                results.append(res)
        except Exception as e:
            name = os.path.basename(top10_csv)
            print(f"\n  [오류] {name} 처리 중 예외 발생: {e}")
            import traceback; traceback.print_exc()
            failed.append({"file": name, "error": str(e)})

    print(f"\n\n{'='*70}")
    print(f"전체 처리 완료: 성공 {len(results)}개 / 실패 {len(failed)}개")
    print(f"{'='*70}")

    if results:
        pd.DataFrame(results).to_csv(
            os.path.join(SAVE_DIR, "전체_업종_스코어_처리요약.csv"),
            index=False, encoding="utf-8-sig"
        )

    if failed:
        print(f"\n[실패 목록]")
        for f in failed:
            print(f"  {f['file']} : {f['error']}")
        pd.DataFrame(failed).to_csv(
            os.path.join(SAVE_DIR, "실패_목록.csv"),
            index=False, encoding="utf-8-sig"
        )


if __name__ == "__main__":
    main()

전체 업종 수: 16개
  M03_음식료품_제조업_Top10 우수모델.csv
  M04_섬유_가죽_신발_제조업_Top10 우수모델.csv
  M08_화학_의약품_고무_플라스틱_제조업_Top10 우수모델.csv
  M10_제1차금속산업_Top10 우수모델.csv
  M11_조립금속제품_제조업_Top10 우수모델.csv
  M12_기타기계장비_제조업_Top10 우수모델.csv
  M13_전자부품_컴퓨터_전기장비_제조업_Top10 우수모델.csv
  M15_운송장비_제조업_Top10 우수모델.csv
  M17_전기_가스_수도사업_Top10 우수모델.csv
  M18_건설업_Top10 우수모델.csv
  M19_도매및소매업_Top10 우수모델.csv
  M20_숙박_음식점업_Top10 우수모델.csv
  M21_운수_창고업_Top10 우수모델.csv
  M22_정보통신업_Top10 우수모델.csv
  M23_부동산_임대_사업서비스업_Top10 우수모델.csv
  M25_오락_문화_개인서비스업_Top10 우수모델.csv

[업종] M03_음식료품_제조업
  lifecycle 파일: M03_음식료품_제조업_lifecycle_scored_yearly_minmax.csv
  부실확률 결측 삭제: 10478 → 7325행 (3153행 제거)
  저장 완료 → 중간결과\22_SCORE\M03_음식료품_제조업_통합_스코어_데이터.csv  (7325행)
  [결측치 비율]
  생애주기점수      0.000
  Porter5F    0.000
  산업충격민감도     0.000
  기업충격민감도     0.005
  부실확률        0.000
  부실확률변화      0.000
  [점수 범위 확인 (min/max)]
       생애주기점수  Porter5F  산업충격민감도  기업충격민감도   부실확률  부실확률변화
  min     0.0       0.0      0.0      0.0    0.0     0.0
  max   100.0     100.0    100.0

### 신용등급 병합 +가중치 그리드 서치 +등급 부여

In [13]:
# -*- coding: utf-8 -*-
# ============================================================
# 전체 업종 스코어 종합 파이프라인
#
# [흐름]
#  1. 통합_스코어_데이터.csv + 신용등급.xlsx 병합
#  2. 그리드서치 → 최적 가중치 도출
#  3. 최종합산스코어 계산
#  4. 분위수 기반 5단계 등급 부여
#  5. 통합_스코어_데이터에 신용등급 컬럼 추가
#  6. 저장
#
# [저장 경로]
#   중간결과\22_SCORE\{prefix}_통합_스코어_데이터.csv         ← 최종합산스코어 + 5단계등급 + 신용등급 추가
#   중간결과\22_SCORE\신용등급병합\{prefix}_신용등급_병합.csv
#   중간결과\22_SCORE\그리드서치\{prefix}_가중치_그리드서치_결과.csv
#   중간결과\22_SCORE\그리드서치\전체_업종_최적가중치_요약.csv
# ============================================================

import re
import os
import glob
import warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")


# ============================================================
# 1. 전역 경로 설정
# ============================================================
BASE_DIR    = r'중간결과'
TOP10_FOLDER = os.path.join(BASE_DIR, r'15_우수모델\Top10')
SCORE_DIR   = os.path.join(BASE_DIR, '22_SCORE')
CREDIT_PATH = r'데이터수집\신용등급\상장사 신용등급.xlsx'

SAVE_MERGE_DIR = os.path.join(SCORE_DIR, '신용등급병합')
SAVE_GRID_DIR  = os.path.join(SCORE_DIR, '그리드서치')

for d in [SAVE_MERGE_DIR, SAVE_GRID_DIR]:
    os.makedirs(d, exist_ok=True)

PD_COL     = "부실확률"
OTHER_COLS = ["Porter5F", "생애주기점수", "산업충격민감도", "기업충격민감도", "부실확률변화"]
ALL_W_COLS = [PD_COL] + OTHER_COLS

# 분위수 등급 설정 (높은 스코어 = 위험)
QUANTILE_CUTS = {
    "매우 위험" : 0.05,
    "위험"      : 0.15,
    "중립"      : 0.30,
    "안정"      : 0.50,
    "매우 안정" : 1.00,
}
GRADE_LABELS = ["매우 안정", "안정", "중립", "위험", "매우 위험"]

GRADE_ORDER = [
    "AAA",
    "AA+", "AA", "AA-",
    "A+",  "A",  "A-",
    "BBB+","BBB","BBB-",
    "BB+", "BB", "BB-",
    "B+",  "B",  "B-",
    "CCC+","CCC","CCC-",
    "CC",  "C",  "D"
]
GRADE_MAP  = {g: i+1 for i, g in enumerate(GRADE_ORDER)}
MANUAL_FIX = {
    "AA+STABLE":  "AA+",
    "AA+POSTIVE": "AA+",
    "AASTABLE":   "AA",
    "AA-STABLE":  "AA-",
    "A+STABLE":   "A+",
}


# ============================================================
# 2. 유틸 함수
# ============================================================

def parse_prefix(top10_csv_path):
    fname = os.path.basename(top10_csv_path)
    match = re.match(r'^(.+?)_Top10', fname)
    if not match:
        raise ValueError(f"파일명 패턴 불일치: {fname}")
    return match.group(1)


def is_long_term(rating):
    if pd.isna(rating):
        return False
    grade = str(rating).split("/")[0]
    return not re.search(r"\d", grade)


def gen_combos(total_units, n_vars):
    if n_vars == 1:
        yield (total_units,)
        return
    for i in range(total_units + 1):
        for rest in gen_combos(total_units - i, n_vars - 1):
            yield (i,) + rest


# ============================================================
# 3. 신용등급 파일 전처리 (전체 공통 1회)
# ============================================================

def load_credit(credit_path):
    print(f"신용등급 파일 로드 중: {credit_path}")
    credit_df = pd.read_excel(credit_path, dtype={"사업자등록번호": str})
    credit_df["사업자등록번호"] = (
        credit_df["사업자등록번호"]
        .str.strip()
        .str.replace("-", "", regex=False)
        .str.zfill(10)
    )

    credit_df["연도"] = credit_df["회계년도"].astype(str).str.split("/").str[0].astype(int)
    credit_df["월"]   = credit_df["회계년도"].astype(str).str.split("/").str[1].astype(int)

    credit_filtered = credit_df[credit_df["평가사구분"].isin([10])].copy()
    credit_filtered = credit_filtered[
        credit_filtered["신용등급"].apply(is_long_term)
    ].copy()

    # 연말(가장 최근 월) 등급 채택
    idx_latest = (
        credit_filtered
        .groupby(["사업자등록번호", "연도"])["월"]
        .idxmax()
    )
    credit_filtered = credit_filtered.loc[idx_latest].copy()

    credit_filtered["등급_base"] = credit_filtered["신용등급"].str.split("/").str[0]
    credit_filtered = credit_filtered.drop_duplicates(
        subset=["사업자등록번호", "연도", "월", "평가사명 및 등급", "신용등급"]
    )

    def collapse_if_same(group):
        unique_base = group["등급_base"].dropna().unique()
        if len(unique_base) <= 1:
            return group.iloc[[0]]
        return group.iloc[[0]]   # 불일치 → 첫 행 선택

    credit_filtered = (
        credit_filtered
        .groupby(["사업자등록번호", "연도"], group_keys=False)
        .apply(collapse_if_same)
        .reset_index(drop=True)
    )

    # 병합용: 연도 → 연도_신용등급으로 rename
    credit_for_merge = credit_filtered[
        ["사업자등록번호", "연도", "신용등급"]
    ].copy().rename(columns={"연도": "연도_신용등급"})

    print(f"  전처리 완료: {len(credit_for_merge)}행 "
          f"| unique 기업: {credit_for_merge['사업자등록번호'].nunique()}개")
    return credit_for_merge


# ============================================================
# 4. 등급 순위 매핑
# ============================================================

def map_grade(df):
    df = df.copy()
    df["등급_정제"] = (
        df["신용등급"]
        .str.split(r"[/(\s]").str[0]
        .str.strip()
        .str.upper()
    )
    df["등급_정제"] = df["등급_정제"].replace(MANUAL_FIX)
    df["등급_순위"] = df["등급_정제"].map(GRADE_MAP)

    unmapped = df[df["등급_순위"].isna()]["등급_정제"].unique()
    if len(unmapped) > 0:
        print(f"  [경고] 매핑 안 된 등급: {unmapped} → 해당 행 제거")

    df = df.dropna(subset=["등급_순위"]).copy()
    df["등급_순위"] = df["등급_순위"].astype(int)
    return df


# ============================================================
# 5. 그리드서치
# ============================================================

def run_gridsearch(merged):
    y       = merged["등급_순위"].values
    n_other = len(OTHER_COLS)
    results = []

    for pd_units in range(10, 15):   # 부실확률 50%~70%
        w_pd        = round(pd_units * 0.05, 2)
        remaining   = 20 - pd_units
        extra_units = remaining - n_other

        if extra_units < 0:
            continue

        for extra_combo in gen_combos(extra_units, n_other):
            weights_other = [round((1 + e) * 0.05, 2) for e in extra_combo]
            total_w = round(w_pd + sum(weights_other), 2)
            if abs(total_w - 1.0) > 0.01:
                continue

            score = merged[PD_COL].values * w_pd
            for col, w in zip(OTHER_COLS, weights_other):
                vals = merged[col].fillna(0).values if col in merged.columns else np.zeros(len(merged))
                score = score + vals * w

            rho, p = spearmanr(score, y)
            results.append({
                "부실확률_w": w_pd,
                **{f"{c}_w": w for c, w in zip(OTHER_COLS, weights_other)},
                "가중치합":       total_w,
                "spearman_rho":  round(rho, 6),
                "p_value":       round(p, 9),   # ★ 소수점 9자리
            })

    res_df = pd.DataFrame(results)
    if res_df.empty:
        return res_df

    res_df["abs_rho"] = res_df["spearman_rho"].abs()
    res_df = res_df.sort_values(
        ["p_value", "abs_rho"], ascending=[True, False]
    ).drop(columns="abs_rho").reset_index(drop=True)
    return res_df


# ============================================================
# 6. 5단계 등급 부여
# ============================================================

def assign_grade(score_series):
    quantiles = [0] + [1 - v for v in list(QUANTILE_CUTS.values())[:-1]] + [1]
    quantiles = sorted(quantiles)
    boundaries = score_series.quantile(quantiles).values

    grade_series = pd.cut(
        score_series,
        bins=boundaries,
        labels=GRADE_LABELS,
        include_lowest=True
    )
    return grade_series, boundaries


# ============================================================
# 7. 단일 업종 처리 함수
# ============================================================

def process_one_industry(top10_csv_path, credit_df):
    prefix = parse_prefix(top10_csv_path)

    print(f"\n{'='*70}")
    print(f"[업종] {prefix}")
    print(f"{'='*70}")

    # ── 스코어 파일 확인
    score_path = os.path.join(SCORE_DIR, f"{prefix}_통합_스코어_데이터.csv")
    if not os.path.exists(score_path):
        print(f"  [경고] 스코어 파일 없음: {score_path} → skip")
        return None

    score_df = pd.read_csv(score_path, dtype={"사업자등록번호": str})
    score_df["사업자등록번호"] = score_df["사업자등록번호"].str.zfill(10)
    print(f"  스코어 데이터: {score_df.shape[0]}행 "
          f"| 기업 수: {score_df['사업자등록번호'].nunique()}개")

    # ──────────────────────────────────────────────────────────
    # Step 1. 신용등급 병합 (score 연도 + 1 = 신용등급 연도)
    # ──────────────────────────────────────────────────────────
    score_df["연도_신용등급"] = score_df["연도"] + 1

    merged = score_df.merge(
        credit_df,
        on=["사업자등록번호", "연도_신용등급"],
        how="inner"
    )
    print(f"  신용등급 병합 후: {len(merged)}행")

    if len(merged) == 0:
        print(f"  [경고] 병합 결과 0행 → 그리드서치 skip")
        best_weights = None
    else:
        # 등급 순위 매핑
        merged = map_grade(merged)
        if len(merged) == 0:
            print(f"  [경고] 등급 매핑 후 0행 → 그리드서치 skip")
            best_weights = None
        else:
            # 신용등급 병합 파일 저장
            merge_save_path = os.path.join(
                SAVE_MERGE_DIR, f"{prefix}_신용등급_병합.csv"
            )
            merged.to_csv(merge_save_path, index=False, encoding="utf-8-sig")
            print(f"  신용등급 병합 저장 → {merge_save_path}")

            # ──────────────────────────────────────────────────
            # Step 2. 그리드서치
            # ──────────────────────────────────────────────────
            if PD_COL not in merged.columns:
                print(f"  [경고] '{PD_COL}' 컬럼 없음 → 그리드서치 skip")
                best_weights = None
            else:
                res_df = run_gridsearch(merged)
                display_cols = (
                    ["부실확률_w"] +
                    [f"{c}_w" for c in OTHER_COLS] +
                    ["가중치합", "spearman_rho", "p_value"]
                )

                sig_count = int((res_df["p_value"] < 0.05).sum()) if not res_df.empty else 0
                print(f"  그리드서치: {len(res_df)}개 조합 | p<0.05: {sig_count}개")

                if not res_df.empty:
                    grid_save_path = os.path.join(
                        SAVE_GRID_DIR, f"{prefix}_가중치_그리드서치_결과.csv"
                    )
                    res_df[display_cols].to_csv(
                        grid_save_path, index=False, encoding="utf-8-sig"
                    )
                    print(f"  그리드서치 저장 → {grid_save_path}")

                    best = res_df.iloc[0]
                    best_weights = {
                        "부실확률_w"   : best["부실확률_w"],
                        **{f"{c}_w": best[f"{c}_w"] for c in OTHER_COLS},
                        "spearman_rho" : best["spearman_rho"],
                        "p_value"      : best["p_value"],
                        "N_Sig_p05"    : sig_count,
                    }
                    print(f"  ★ 최적: 부실확률_w={best['부실확률_w']:.2f} "
                          f"| rho={best['spearman_rho']:.6f} "
                          f"| p={best['p_value']:.9f}")
                else:
                    best_weights = None

    # ──────────────────────────────────────────────────────────
    # Step 3. 최종합산스코어 계산
    # ──────────────────────────────────────────────────────────
    if best_weights is not None:
        w_pd = best_weights["부실확률_w"]
        score_df["최종합산스코어"] = score_df[PD_COL].fillna(0) * w_pd
        for col in OTHER_COLS:
            w = best_weights[f"{col}_w"]
            vals = score_df[col].fillna(0) if col in score_df.columns else 0
            score_df["최종합산스코어"] += vals * w
        score_df["최종합산스코어"] = score_df["최종합산스코어"].round(1)
        print(f"  최종합산스코어 계산 완료 "
              f"(범위: {score_df['최종합산스코어'].min():.1f} ~ "
              f"{score_df['최종합산스코어'].max():.1f})")
    else:
        # 그리드서치 실패 시 균등 가중치로 fallback
        print(f"  [참고] 그리드서치 없음 → 균등 가중치(1/6) 적용")
        w_equal = round(1 / len(ALL_W_COLS), 4)
        score_df["최종합산스코어"] = sum(
            score_df[c].fillna(0) * w_equal
            for c in ALL_W_COLS if c in score_df.columns
        )
        score_df["최종합산스코어"] = score_df["최종합산스코어"].round(1)

    # ──────────────────────────────────────────────────────────
    # Step 4. 5단계 등급 부여
    # ──────────────────────────────────────────────────────────
    grade_series, boundaries = assign_grade(score_df["최종합산스코어"])
    score_df["등급_5단계"] = grade_series

    print(f"\n  [분위수 기반 점수 경계값]")
    for i, label in enumerate(GRADE_LABELS):
        print(f"    {label:8s}: {boundaries[i]:.2f} ~ {boundaries[i+1]:.2f}")

    # 검증 리포트
    report = score_df.groupby("등급_5단계", observed=True).agg(
        최소점수=("최종합산스코어", "min"),
        최대점수=("최종합산스코어", "max"),
        행수=("부실라벨_ICR3년", "count"),
        실제부실수=("부실라벨_ICR3년", "sum"),
        구간내_부실률=("부실라벨_ICR3년", "mean")
    ).reindex(["매우 위험", "위험", "중립", "안정", "매우 안정"])
    report["구간내_부실률"] = report["구간내_부실률"].map("{:.2%}".format)
    print(f"\n  [등급별 검증 리포트]")
    print("  " + report.to_string().replace("\n", "\n  "))

    # ──────────────────────────────────────────────────────────
    # Step 5. 통합_스코어_데이터에 신용등급 컬럼 추가
    # ──────────────────────────────────────────────────────────
    if len(merged) > 0:
        rating_subset = merged[["사업자등록번호", "연도", "신용등급"]].drop_duplicates()
        score_df = pd.merge(
            score_df,
            rating_subset,
            on=["사업자등록번호", "연도"],
            how="left"
        )
        print(f"\n  신용등급 컬럼 추가 완료 "
              f"(등급 있는 행: {score_df['신용등급'].notna().sum()}개)")

    # ──────────────────────────────────────────────────────────
    # Step 6. 저장
    # ──────────────────────────────────────────────────────────
    score_df.to_csv(score_path, index=False, encoding="utf-8-sig")
    print(f"\n  통합_스코어_데이터 저장 완료 → {score_path}")

    result_row = {
        "Industry"     : prefix,
        "N_Score"      : len(score_df),
        "N_Merged"     : len(merged) if len(merged) > 0 else 0,
        "Score_Min"    : score_df["최종합산스코어"].min(),
        "Score_Max"    : score_df["최종합산스코어"].max(),
    }
    if best_weights:
        result_row.update({
            "Best_부실확률_w"   : best_weights["부실확률_w"],
            "Best_rho"         : best_weights["spearman_rho"],
            "Best_p"           : best_weights["p_value"],
            "N_Sig_p05"        : best_weights["N_Sig_p05"],
        })
    return result_row


# ============================================================
# 8. 전체 Top10 파일 순회 실행
# ============================================================

def main():
    top10_files = sorted(glob.glob(os.path.join(TOP10_FOLDER, "*.csv")))
    if not top10_files:
        print(f"[오류] Top10 파일이 없습니다: {TOP10_FOLDER}")
        return

    # 신용등급 파일 1회 로드
    if not os.path.exists(CREDIT_PATH):
        print(f"[오류] 신용등급 파일 없음: {CREDIT_PATH}")
        return
    credit_df = load_credit(CREDIT_PATH)

    print(f"\n전체 업종 수: {len(top10_files)}개")
    for f in top10_files:
        print(f"  {os.path.basename(f)}")

    results = []
    failed  = []

    for top10_csv in top10_files:
        try:
            res = process_one_industry(top10_csv, credit_df)
            if res:
                results.append(res)
        except Exception as e:
            name = os.path.basename(top10_csv)
            print(f"\n  [오류] {name} 처리 중 예외: {e}")
            import traceback; traceback.print_exc()
            failed.append({"file": name, "error": str(e)})

    # 전체 최적 가중치 요약 저장
    print(f"\n\n{'='*70}")
    print(f"전체 처리 완료: 성공 {len(results)}개 / 실패 {len(failed)}개")
    print(f"{'='*70}")

    if results:
        summary_df = pd.DataFrame(results)
        summary_path = os.path.join(SAVE_GRID_DIR, "전체_업종_최적가중치_요약.csv")
        summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
        print(f"\n전체 요약 저장 → {summary_path}")
        show_cols = [c for c in ["Industry","N_Score","N_Merged","Best_rho","Best_p"] if c in summary_df.columns]
        print(summary_df[show_cols].to_string(index=False))

    if failed:
        print(f"\n[실패 목록]")
        for f in failed:
            print(f"  {f['file']} : {f['error']}")
        pd.DataFrame(failed).to_csv(
            os.path.join(SAVE_GRID_DIR, "실패_목록.csv"),
            index=False, encoding="utf-8-sig"
        )


if __name__ == "__main__":
    main()

신용등급 파일 로드 중: 데이터수집\신용등급\상장사 신용등급.xlsx
  전처리 완료: 2068행 | unique 기업: 372개

전체 업종 수: 16개
  M03_음식료품_제조업_Top10 우수모델.csv
  M04_섬유_가죽_신발_제조업_Top10 우수모델.csv
  M08_화학_의약품_고무_플라스틱_제조업_Top10 우수모델.csv
  M10_제1차금속산업_Top10 우수모델.csv
  M11_조립금속제품_제조업_Top10 우수모델.csv
  M12_기타기계장비_제조업_Top10 우수모델.csv
  M13_전자부품_컴퓨터_전기장비_제조업_Top10 우수모델.csv
  M15_운송장비_제조업_Top10 우수모델.csv
  M17_전기_가스_수도사업_Top10 우수모델.csv
  M18_건설업_Top10 우수모델.csv
  M19_도매및소매업_Top10 우수모델.csv
  M20_숙박_음식점업_Top10 우수모델.csv
  M21_운수_창고업_Top10 우수모델.csv
  M22_정보통신업_Top10 우수모델.csv
  M23_부동산_임대_사업서비스업_Top10 우수모델.csv
  M25_오락_문화_개인서비스업_Top10 우수모델.csv

[업종] M03_음식료품_제조업
  스코어 데이터: 7325행 | 기업 수: 1164개
  신용등급 병합 후: 121행
  신용등급 병합 저장 → 중간결과\22_SCORE\신용등급병합\M03_음식료품_제조업_신용등급_병합.csv
  그리드서치: 251개 조합 | p<0.05: 190개
  그리드서치 저장 → 중간결과\22_SCORE\그리드서치\M03_음식료품_제조업_가중치_그리드서치_결과.csv
  ★ 최적: 부실확률_w=0.50 | rho=0.459654 | p=0.000000114
  최종합산스코어 계산 완료 (범위: 2.9 ~ 93.9)

  [분위수 기반 점수 경계값]
    매우 안정   : 2.90 ~ 14.20
    안정      : 14.20 ~ 18.10
    중립      : 18.10 ~ 32.70

# 대시보드

In [1]:
# -*- coding: utf-8 -*-
# ============================================================
# 전체 업종 대시보드 데이터 생성
#
# [흐름]
#  1. 통합_스코어_데이터 로드 → 2023 정상기업 필터링 (2022~2024)
#  2. 모델 재학습 (Top10 1위 조합)
#  3. 전체 데이터 SHAP 계산 (Waterfall용 개별 SHAP, 모델별 Explainer 분기)
#  4. Top10 SHAP 변수 추출 → 기업별 병합
#  5. feature_analysis CSV → 핵심피처 추출 (산업 파일)
#
# [저장 경로]
#   중간결과\23_대시보드\{prefix}_기업.csv
#   중간결과\23_대시보드\{prefix}_산업.csv
# ============================================================

import os
import re
import gc
import glob
import warnings
import numpy as np
import pandas as pd
import shap
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score

from xgboost import XGBClassifier

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

try:
    from ctgan import CTGAN
    HAS_CTGAN = True
except ImportError:
    HAS_CTGAN = False

warnings.filterwarnings("ignore")


# ============================================================
# 1. 전역 경로 설정
# ============================================================
BASE_DIR       = r'중간결과'
TOP10_FOLDER   = os.path.join(BASE_DIR, r'15_우수모델\Top10')
TRAIN_FOLDER   = os.path.join(BASE_DIR, '12_train')
TEST_FOLDER    = os.path.join(BASE_DIR, '12_test')
FEATURE_FOLDER = os.path.join(BASE_DIR, '13_피처셀렉션')
SHAP_DIR       = os.path.join(BASE_DIR, '16_SHAP')
SCORE_DIR      = os.path.join(BASE_DIR, '22_SCORE')
DASHBOARD_DIR  = os.path.join(BASE_DIR, '23_대시보드')

os.makedirs(DASHBOARD_DIR, exist_ok=True)

TARGET_COL   = "부실라벨_ICR3년"
YEAR_COL     = "회계년도"
COMPANY_COL  = "회사명"
ID_COL       = "사업자등록번호"
RANDOM_STATE = 42
RECALL_MIN   = 0.9
N_TOP        = 10   # SHAP 상위 N개

FILTER_YEARS    = [2022, 2023, 2024]
FILTER_BASE_YEAR = 2023   # 이 연도에 정상(0)인 기업만


# ============================================================
# 2. 유틸 함수
# ============================================================

def parse_industry_info(top10_csv_path):
    fname = os.path.basename(top10_csv_path)
    match = re.match(r'^(.+?)_Top10', fname)
    if not match:
        raise ValueError(f"파일명 패턴 불일치: {fname}")
    prefix = match.group(1)
    return {
        "prefix"         : prefix,
        "train_file"     : os.path.join(TRAIN_FOLDER, f"{prefix}_train.parquet"),
        "test_file"      : os.path.join(TEST_FOLDER,  f"{prefix}_test.parquet"),
        "feature_folder" : os.path.join(FEATURE_FOLDER, prefix),
    }


def _parse_smote_ratio(smote_ratio):
    try:
        if smote_ratio is None:
            return None
        s = str(smote_ratio).strip()
        if s in ("", "-", "nan", "None"):
            return None
        return float(s)
    except (TypeError, ValueError):
        return None


def _ctgan_resample(X, y, ratio):
    if not HAS_CTGAN:
        raise ImportError("pip install ctgan")
    if not isinstance(X, pd.DataFrame):
        raise TypeError("X는 pd.DataFrame이어야 합니다.")
    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)
    n_minority = int((y == 1).sum())
    n_majority = int((y == 0).sum())
    if n_minority < 5:
        return X, y
    target_minority = int(n_majority * ratio) if ratio is not None else n_majority
    n_synth = target_minority - n_minority
    if n_synth <= 0:
        return X, y
    minority_df = X[y == 1].copy().reset_index(drop=True)
    try:
        ctgan = CTGAN(epochs=300, verbose=False)
    except TypeError:
        ctgan = CTGAN(epochs=300)
    ctgan.fit(minority_df)
    synth = ctgan.sample(n_synth).reindex(columns=X.columns)
    X_res = pd.concat([X, synth], ignore_index=True)
    y_res = pd.concat([y, pd.Series([1] * n_synth)], ignore_index=True)
    return X_res, y_res


def apply_resampling(X, y, method, smote_ratio):
    method_l = str(method).strip().lower()
    ratio    = _parse_smote_ratio(smote_ratio)
    if method_l == "classweight":
        pos_weight = (y == 0).sum() / (y == 1).sum()
        return X, y, pos_weight
    if method_l in ("none", "", "-", "nan"):
        return X, y, 1.0
    if "borderline" in method_l:
        if not HAS_IMBLEARN:
            raise ImportError("pip install imbalanced-learn")
        sampler = BorderlineSMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0
    if method_l == "smote":
        if not HAS_IMBLEARN:
            raise ImportError("pip install imbalanced-learn")
        sampler = SMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0
    if "ctgan" in method_l:
        X_res, y_res = _ctgan_resample(X, y, ratio)
        return X_res, y_res, 1.0
    print(f"  [경고] 알 수 없는 Method='{method}' → ClassWeight로 처리")
    pos_weight = (y == 0).sum() / (y == 1).sum()
    return X, y, pos_weight


def make_model(model_name, pos_weight):
    mn = str(model_name).strip()
    if mn == "XGBoost":
        return XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8, eval_metric="aucpr",
            random_state=RANDOM_STATE, verbosity=0, scale_pos_weight=pos_weight
        )
    if mn == "LightGBM":
        if not HAS_LGBM:
            raise ImportError("pip install lightgbm")
        if pos_weight > 1.0:
            return LGBMClassifier(
                n_estimators=300, learning_rate=0.05, max_depth=4,
                subsample=0.8, colsample_bytree=0.8,
                random_state=RANDOM_STATE, verbose=-1, scale_pos_weight=pos_weight
            )
        return LGBMClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8,
            random_state=RANDOM_STATE, verbose=-1
        )
    if mn == "RandomForest":
        if pos_weight > 1.0:
            return RandomForestClassifier(
                n_estimators=300, max_depth=4, min_samples_leaf=5,
                random_state=RANDOM_STATE, n_jobs=1, class_weight="balanced"
            )
        return RandomForestClassifier(
            n_estimators=300, max_depth=4, min_samples_leaf=5,
            random_state=RANDOM_STATE, n_jobs=1
        )
    if mn == "LogisticRegression":
        if pos_weight > 1.0:
            return LogisticRegression(
                penalty="l2", C=1.0, solver="lbfgs",
                max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"
            )
        return LogisticRegression(
            penalty="l2", C=1.0, solver="lbfgs",
            max_iter=1000, random_state=RANDOM_STATE
        )
    raise ValueError(f"지원하지 않는 모델명: {model_name}")


def find_threshold_at_recall(y_true, y_prob, recall_min=RECALL_MIN):
    thresholds = np.arange(0.01, 1.0, 0.01)
    valid = [round(thr, 2) for thr in thresholds
             if recall_score(y_true, (y_prob >= thr).astype(int), zero_division=0) >= recall_min]
    return max(valid) if valid else None


# ============================================================
# 3. 모델별 SHAP Explainer (Waterfall용 → explainer() 객체 방식)
# ============================================================

def get_shap_explanation(model, model_name, X):
    """
    Waterfall 호환: shap.Explanation 객체 반환
    - Tree 계열: TreeExplainer → explainer(X)
    - LogisticRegression: LinearExplainer → explainer(X)
    shap_values는 항상 2D (n_samples, n_features) 보장
    """
    mn = str(model_name).strip()

    if mn in ("XGBoost", "LightGBM", "RandomForest"):
        explainer = shap.TreeExplainer(model)
        sv = explainer(X, check_additivity=False)

        # shap.Explanation 내 values 정규화
        vals = sv.values
        if isinstance(vals, list):
            vals = vals[1]
        vals = np.array(vals)
        if vals.ndim == 3:
            vals = vals[:, :, 1]
        if vals.ndim == 1:
            vals = vals.reshape(1, -1)

        # base_values 정규화
        bv = sv.base_values
        if isinstance(bv, np.ndarray) and bv.ndim == 2:
            bv = bv[:, 1]
        elif isinstance(bv, np.ndarray) and bv.ndim == 0:
            bv = np.full(len(X), float(bv))

        return vals, bv

    if mn == "LogisticRegression":
        explainer = shap.LinearExplainer(model, X)
        sv = explainer(X)
        vals = sv.values
        if isinstance(vals, list):
            vals = vals[1]
        vals = np.array(vals)
        if vals.ndim == 3:
            vals = vals[:, :, 1]
        bv = sv.base_values
        if isinstance(bv, np.ndarray) and bv.ndim == 2:
            bv = bv[:, 1]
        return vals, bv

    # fallback: KernelExplainer (느림, 샘플 제한)
    print(f"  [SHAP] {mn} → KernelExplainer (샘플 200개)")
    bg  = shap.sample(X, min(100, len(X)))
    exp = shap.KernelExplainer(model.predict_proba, bg)
    sv  = exp.shap_values(X.iloc[:min(200, len(X))])
    vals = sv[1] if isinstance(sv, list) else sv
    bv   = np.full(len(vals), exp.expected_value[1] if isinstance(exp.expected_value, (list, np.ndarray)) else exp.expected_value)
    return np.array(vals), bv


# ============================================================
# 4. 단일 업종 처리
# ============================================================

def process_one_industry(top10_csv_path):
    info   = parse_industry_info(top10_csv_path)
    prefix = info["prefix"]

    print(f"\n{'='*70}")
    print(f"[업종] {prefix}")
    print(f"{'='*70}")

    # ── Top10 1위 행 로드
    top10 = pd.read_csv(top10_csv_path, index_col=0)
    if len(top10) == 0:
        print(f"  [경고] Top10 파일 비어있음 → skip")
        return None

    row          = top10.iloc[0]
    FEATURE_SET  = row["FeatureSet"]
    FEATURE_FILE = row["FeatureFile"]
    METHOD       = row["Method"]
    SMOTE_RATIO  = row["SMOTE_Ratio"]
    MODEL_NAME   = row["Model"]

    print(f"  Model={MODEL_NAME} | Method={METHOD} | FeatureSet={FEATURE_SET}")

    # ── 파일 존재 확인
    feature_path = os.path.join(info["feature_folder"], FEATURE_FILE)
    score_path   = os.path.join(SCORE_DIR, f"{prefix}_통합_스코어_데이터.csv")
    shap_feat_path = os.path.join(
        SHAP_DIR, prefix, f"feature_analysis_{FEATURE_SET}.csv"
    )

    for label, path in [
        ("Train",        info["train_file"]),
        ("Test",         info["test_file"]),
        ("피처 파일",    feature_path),
        ("스코어 데이터", score_path),
    ]:
        if not os.path.exists(path):
            print(f"  [경고] {label} 없음: {path} → skip")
            return None

    # ── 데이터 로드
    train_full = pd.read_parquet(info["train_file"])
    test       = pd.read_parquet(info["test_file"])
    y_train    = train_full[TARGET_COL]
    y_test     = test[TARGET_COL]
    all_data   = pd.concat([train_full, test], ignore_index=True)

    feat_df      = pd.read_csv(feature_path)
    col_key      = "feature" if "feature" in feat_df.columns else feat_df.columns[0]
    use_features = [f for f in feat_df[col_key].tolist() if f in train_full.columns]
    print(f"  피처 수: {len(use_features)}개 | 전체 데이터: {len(all_data)}행")

    # ── 결측치 처리
    imputer     = SimpleImputer(strategy="median")
    X_train_imp = pd.DataFrame(
        imputer.fit_transform(train_full[use_features]), columns=use_features
    )
    X_test_imp  = pd.DataFrame(
        imputer.transform(test[use_features]), columns=use_features
    )

    # ── 리샘플링 + 모델 학습
    X_res, y_res, pos_weight = apply_resampling(X_train_imp, y_train, METHOD, SMOTE_RATIO)
    model = make_model(MODEL_NAME, pos_weight)
    model.fit(X_res, y_res)
    print(f"  모델 학습 완료 ({MODEL_NAME})")

    # ── Threshold 산출 (Test 기준 Recall >= RECALL_MIN)
    y_prob_test = model.predict_proba(X_test_imp)[:, 1]
    found_thr   = find_threshold_at_recall(y_test, y_prob_test, RECALL_MIN)
    THRESHOLD   = found_thr if found_thr is not None else 0.01
    print(f"  Threshold (Recall>={RECALL_MIN}) = {THRESHOLD:.2f}")

    # ──────────────────────────────────────────────────────────
    # Step A. 통합_스코어_데이터 기반 필터링
    #   2023년 정상(0)인 기업의 2022~2024년 데이터
    # ──────────────────────────────────────────────────────────
    score_df = pd.read_csv(score_path, dtype={ID_COL: str})
    score_df[ID_COL] = score_df[ID_COL].str.zfill(10)

    valid_companies = score_df[
        score_df["연도"] == FILTER_BASE_YEAR
    ].query(f"{TARGET_COL} == 0")[ID_COL].unique()

    filtered_df = score_df[
        score_df[ID_COL].isin(valid_companies) &
        score_df["연도"].isin(FILTER_YEARS)
    ].copy()
    print(f"  필터링: {len(filtered_df)}행 "
          f"(기업 수: {filtered_df[ID_COL].nunique()}개, "
          f"연도: {FILTER_YEARS})")

    # ──────────────────────────────────────────────────────────
    # Step B. 전체 데이터 SHAP 계산 (Waterfall용 개별값)
    # ──────────────────────────────────────────────────────────
    all_feat_imp = pd.DataFrame(
        imputer.transform(all_data[use_features]),
        columns=use_features,
        index=all_data.index
    )

    print(f"  SHAP 계산 중... (전체 {len(all_data)}행)")
    shap_vals, base_vals = get_shap_explanation(model, MODEL_NAME, all_feat_imp)
    print(f"  SHAP 완료. shape={shap_vals.shape}")

    y_prob_all = model.predict_proba(all_feat_imp)[:, 1]
    y_pred_all = (y_prob_all >= THRESHOLD).astype(int)
    feature_arr = np.array(use_features)

    # ── Top N SHAP 추출
    n_rows      = len(all_data)
    top_feature = np.empty((n_rows, N_TOP), dtype=object)
    top_value   = np.full((n_rows, N_TOP), np.nan)
    top_shap    = np.full((n_rows, N_TOP), np.nan)

    for i in range(n_rows):
        order = np.argsort(-np.abs(shap_vals[i]))[:N_TOP]
        k = len(order)
        top_feature[i, :k] = feature_arr[order]
        top_value[i, :k]   = all_feat_imp.iloc[i][feature_arr[order]].values
        top_shap[i, :k]    = shap_vals[i][order]

    # ── shap_summary 생성
    shap_summary = pd.DataFrame({
        ID_COL      : all_data[ID_COL].values if ID_COL in all_data.columns else "",
        COMPANY_COL : all_data[COMPANY_COL].values,
        YEAR_COL    : all_data[YEAR_COL].values,
        "base_value": base_vals,
        "threshold" : THRESHOLD,
        "pred_prob" : y_prob_all,
        "pred_label": y_pred_all,
    })
    for r in range(N_TOP):
        shap_summary[f"Top{r+1}_Feature"]    = top_feature[:, r]
        shap_summary[f"Top{r+1}_Value"]      = top_value[:, r]
        shap_summary[f"Top{r+1}_SHAP_Value"] = top_shap[:, r]

    # 중복 제거 (사업자등록번호 + 회계년도 기준)
    id_col_used = ID_COL if ID_COL in all_data.columns else COMPANY_COL
    shap_summary = shap_summary.drop_duplicates(
        subset=[id_col_used, YEAR_COL], keep="first"
    ).reset_index(drop=True)

    # ──────────────────────────────────────────────────────────
    # Step C. filtered_df + shap_summary 병합 → 기업 파일
    # ──────────────────────────────────────────────────────────
    # filtered_df의 "연도" ↔ shap_summary의 YEAR_COL(회계년도) 매칭
    shap_for_merge = shap_summary.rename(columns={YEAR_COL: "연도"})
    merge_keys = [ID_COL, "연도"] if ID_COL in filtered_df.columns else [COMPANY_COL, "연도"]

    firm_df = filtered_df.merge(
        shap_for_merge,
        on=merge_keys,
        how="left",
        suffixes=("", "_shap")
    )

    # 중복 컬럼 정리
    dup_cols = [c for c in firm_df.columns if c.endswith("_shap")]
    firm_df  = firm_df.drop(columns=dup_cols)

    n_matched   = firm_df["pred_prob"].notna().sum()
    n_unmatched = firm_df["pred_prob"].isna().sum()
    print(f"  SHAP 병합 완료: 매칭 {n_matched}행 / 미매칭 {n_unmatched}행")

    firm_save_path = os.path.join(DASHBOARD_DIR, f"{prefix}_기업.csv")
    firm_df.to_csv(firm_save_path, index=False, encoding="utf-8-sig")
    print(f"  기업 파일 저장 → {firm_save_path}")

    # ──────────────────────────────────────────────────────────
    # Step D. feature_analysis → 핵심피처 추출 → 산업 파일
    # ──────────────────────────────────────────────────────────
    if os.path.exists(shap_feat_path):
        feat_analysis = pd.read_csv(shap_feat_path)
        core_df = feat_analysis[
            feat_analysis["Feature_Type"] == "★ 핵심피처 (SHAP+Perm 모두 높음)"
        ][["Combined_Rank", "Feature", "Category", "Feature_Type"]].reset_index(drop=True)
        print(f"  핵심피처: {len(core_df)}개 추출")
    else:
        print(f"  [경고] feature_analysis 없음: {shap_feat_path} → 빈 산업 파일 생성")
        core_df = pd.DataFrame(
            columns=["Combined_Rank", "Feature", "Category", "Feature_Type"]
        )

    # 업종 정보 추가
    core_df.insert(0, "업종", prefix)
    industry_save_path = os.path.join(DASHBOARD_DIR, f"{prefix}_산업.csv")
    core_df.to_csv(industry_save_path, index=False, encoding="utf-8-sig")
    print(f"  산업 파일 저장 → {industry_save_path}")

    # 메모리 해제
    del train_full, test, all_data, all_feat_imp, shap_vals, model
    gc.collect()

    return {
        "Industry"    : prefix,
        "Model"       : MODEL_NAME,
        "Threshold"   : THRESHOLD,
        "N_Filtered"  : len(filtered_df),
        "N_SHAP_Match": n_matched,
        "N_CoreFeature": len(core_df),
    }


# ============================================================
# 5. 전체 Top10 파일 순회 실행
# ============================================================

def main():
    top10_files = sorted(glob.glob(os.path.join(TOP10_FOLDER, "*.csv")))
    if not top10_files:
        print(f"[오류] Top10 파일이 없습니다: {TOP10_FOLDER}")
        return

    print(f"전체 업종 수: {len(top10_files)}개")
    for f in top10_files:
        print(f"  {os.path.basename(f)}")

    results = []
    failed  = []

    for top10_csv in top10_files:
        try:
            res = process_one_industry(top10_csv)
            if res:
                results.append(res)
        except Exception as e:
            name = os.path.basename(top10_csv)
            print(f"\n  [오류] {name} 처리 중 예외: {e}")
            import traceback; traceback.print_exc()
            failed.append({"file": name, "error": str(e)})

    print(f"\n\n{'='*70}")
    print(f"전체 처리 완료: 성공 {len(results)}개 / 실패 {len(failed)}개")
    print(f"{'='*70}")

    if results:
        pd.DataFrame(results).to_csv(
            os.path.join(DASHBOARD_DIR, "전체_업종_대시보드_처리요약.csv"),
            index=False, encoding="utf-8-sig"
        )

    if failed:
        for f in failed:
            print(f"  {f['file']} : {f['error']}")
        pd.DataFrame(failed).to_csv(
            os.path.join(DASHBOARD_DIR, "실패_목록.csv"),
            index=False, encoding="utf-8-sig"
        )


if __name__ == "__main__":
    main()

전체 업종 수: 16개
  M03_음식료품_제조업_Top10 우수모델.csv
  M04_섬유_가죽_신발_제조업_Top10 우수모델.csv
  M08_화학_의약품_고무_플라스틱_제조업_Top10 우수모델.csv
  M10_제1차금속산업_Top10 우수모델.csv
  M11_조립금속제품_제조업_Top10 우수모델.csv
  M12_기타기계장비_제조업_Top10 우수모델.csv
  M13_전자부품_컴퓨터_전기장비_제조업_Top10 우수모델.csv
  M15_운송장비_제조업_Top10 우수모델.csv
  M17_전기_가스_수도사업_Top10 우수모델.csv
  M18_건설업_Top10 우수모델.csv
  M19_도매및소매업_Top10 우수모델.csv
  M20_숙박_음식점업_Top10 우수모델.csv
  M21_운수_창고업_Top10 우수모델.csv
  M22_정보통신업_Top10 우수모델.csv
  M23_부동산_임대_사업서비스업_Top10 우수모델.csv
  M25_오락_문화_개인서비스업_Top10 우수모델.csv

[업종] M03_음식료품_제조업
  Model=LightGBM | Method=BorderlineSMOTE | FeatureSet=top65_dedup47
  피처 수: 47개 | 전체 데이터: 8885행
  모델 학습 완료 (LightGBM)
  Threshold (Recall>=0.9) = 0.04
  필터링: 2438행 (기업 수: 820개, 연도: [2022, 2023, 2024])
  SHAP 계산 중... (전체 8885행)
  SHAP 완료. shape=(8885, 47)
  SHAP 병합 완료: 매칭 2438행 / 미매칭 0행
  기업 파일 저장 → 중간결과\23_대시보드\M03_음식료품_제조업_기업.csv
  핵심피처: 7개 추출
  산업 파일 저장 → 중간결과\23_대시보드\M03_음식료품_제조업_산업.csv

[업종] M04_섬유_가죽_신발_제조업
  Model=LightGBM | Method=ClassWeight | Feature

In [3]:
# -*- coding: utf-8 -*-
# ============================================================
# 전체 업종 대시보드 데이터 생성
#
# [흐름]
#  1. 통합_스코어_데이터 로드 → 2023 정상기업 필터링 (2022~2024)
#  2. 모델 재학습 (Top10 1위 조합)
#  3. 전체 데이터 SHAP 계산 (Waterfall용 개별 SHAP, 모델별 Explainer 분기)
#  4. Top10 SHAP 변수 추출 → 기업별 병합
#  5. feature_analysis CSV → 핵심피처 추출 (산업 파일)
#
# [저장 경로]
#   중간결과\23_대시보드\{prefix}_기업.csv
#   중간결과\23_대시보드\{prefix}_산업.csv
# ============================================================

import os
import re
import gc
import glob
import warnings
import numpy as np
import pandas as pd
import shap
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score

from xgboost import XGBClassifier

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

try:
    from ctgan import CTGAN
    HAS_CTGAN = True
except ImportError:
    HAS_CTGAN = False

warnings.filterwarnings("ignore")


# ============================================================
# 1. 전역 경로 설정
# ============================================================
BASE_DIR       = r'중간결과'
TOP10_FOLDER   = os.path.join(BASE_DIR, r'15_우수모델\Top10')
TRAIN_FOLDER   = os.path.join(BASE_DIR, '12_train')
TEST_FOLDER    = os.path.join(BASE_DIR, '12_test')
FEATURE_FOLDER = os.path.join(BASE_DIR, '13_피처셀렉션')
SHAP_DIR       = os.path.join(BASE_DIR, '16_SHAP')
SCORE_DIR      = os.path.join(BASE_DIR, '22_SCORE')
DASHBOARD_DIR  = os.path.join(BASE_DIR, '23_대시보드')

os.makedirs(DASHBOARD_DIR, exist_ok=True)

TARGET_COL   = "부실라벨_ICR3년"
YEAR_COL     = "회계년도"
COMPANY_COL  = "회사명"
ID_COL       = "사업자등록번호"
RANDOM_STATE = 42
RECALL_MIN   = 0.9
N_TOP        = 10   # SHAP 상위 N개

FILTER_YEARS    = [2022, 2023, 2024]
FILTER_BASE_YEAR = 2023   # 이 연도에 정상(0)인 기업만


# ============================================================
# 2. 유틸 함수
# ============================================================

def parse_industry_info(top10_csv_path):
    fname = os.path.basename(top10_csv_path)
    match = re.match(r'^(.+?)_Top10', fname)
    if not match:
        raise ValueError(f"파일명 패턴 불일치: {fname}")
    prefix = match.group(1)
    return {
        "prefix"         : prefix,
        "train_file"     : os.path.join(TRAIN_FOLDER, f"{prefix}_train.parquet"),
        "test_file"      : os.path.join(TEST_FOLDER,  f"{prefix}_test.parquet"),
        "feature_folder" : os.path.join(FEATURE_FOLDER, prefix),
    }


def _parse_smote_ratio(smote_ratio):
    try:
        if smote_ratio is None:
            return None
        s = str(smote_ratio).strip()
        if s in ("", "-", "nan", "None"):
            return None
        return float(s)
    except (TypeError, ValueError):
        return None


def _ctgan_resample(X, y, ratio):
    if not HAS_CTGAN:
        raise ImportError("pip install ctgan")
    if not isinstance(X, pd.DataFrame):
        raise TypeError("X는 pd.DataFrame이어야 합니다.")
    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)
    n_minority = int((y == 1).sum())
    n_majority = int((y == 0).sum())
    if n_minority < 5:
        return X, y
    target_minority = int(n_majority * ratio) if ratio is not None else n_majority
    n_synth = target_minority - n_minority
    if n_synth <= 0:
        return X, y
    minority_df = X[y == 1].copy().reset_index(drop=True)
    try:
        ctgan = CTGAN(epochs=300, verbose=False)
    except TypeError:
        ctgan = CTGAN(epochs=300)
    ctgan.fit(minority_df)
    synth = ctgan.sample(n_synth).reindex(columns=X.columns)
    X_res = pd.concat([X, synth], ignore_index=True)
    y_res = pd.concat([y, pd.Series([1] * n_synth)], ignore_index=True)
    return X_res, y_res


def apply_resampling(X, y, method, smote_ratio):
    method_l = str(method).strip().lower()
    ratio    = _parse_smote_ratio(smote_ratio)
    if method_l == "classweight":
        pos_weight = (y == 0).sum() / (y == 1).sum()
        return X, y, pos_weight
    if method_l in ("none", "", "-", "nan"):
        return X, y, 1.0
    if "borderline" in method_l:
        if not HAS_IMBLEARN:
            raise ImportError("pip install imbalanced-learn")
        sampler = BorderlineSMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0
    if method_l == "smote":
        if not HAS_IMBLEARN:
            raise ImportError("pip install imbalanced-learn")
        sampler = SMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0
    if "ctgan" in method_l:
        X_res, y_res = _ctgan_resample(X, y, ratio)
        return X_res, y_res, 1.0
    print(f"  [경고] 알 수 없는 Method='{method}' → ClassWeight로 처리")
    pos_weight = (y == 0).sum() / (y == 1).sum()
    return X, y, pos_weight


def make_model(model_name, pos_weight):
    mn = str(model_name).strip()
    if mn == "XGBoost":
        return XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8, eval_metric="aucpr",
            random_state=RANDOM_STATE, verbosity=0, scale_pos_weight=pos_weight
        )
    if mn == "LightGBM":
        if not HAS_LGBM:
            raise ImportError("pip install lightgbm")
        if pos_weight > 1.0:
            return LGBMClassifier(
                n_estimators=300, learning_rate=0.05, max_depth=4,
                subsample=0.8, colsample_bytree=0.8,
                random_state=RANDOM_STATE, verbose=-1, scale_pos_weight=pos_weight
            )
        return LGBMClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8,
            random_state=RANDOM_STATE, verbose=-1
        )
    if mn == "RandomForest":
        if pos_weight > 1.0:
            return RandomForestClassifier(
                n_estimators=300, max_depth=4, min_samples_leaf=5,
                random_state=RANDOM_STATE, n_jobs=1, class_weight="balanced"
            )
        return RandomForestClassifier(
            n_estimators=300, max_depth=4, min_samples_leaf=5,
            random_state=RANDOM_STATE, n_jobs=1
        )
    if mn == "LogisticRegression":
        if pos_weight > 1.0:
            return LogisticRegression(
                penalty="l2", C=1.0, solver="lbfgs",
                max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"
            )
        return LogisticRegression(
            penalty="l2", C=1.0, solver="lbfgs",
            max_iter=1000, random_state=RANDOM_STATE
        )
    raise ValueError(f"지원하지 않는 모델명: {model_name}")


def find_threshold_at_recall(y_true, y_prob, recall_min=RECALL_MIN):
    thresholds = np.arange(0.01, 1.0, 0.01)
    valid = [round(thr, 2) for thr in thresholds
             if recall_score(y_true, (y_prob >= thr).astype(int), zero_division=0) >= recall_min]
    return max(valid) if valid else None


# ============================================================
# 3. 모델별 SHAP Explainer (Waterfall용 → explainer() 객체 방식)
# ============================================================

def get_shap_explanation(model, model_name, X):
    """
    Waterfall 호환: shap.Explanation 객체 반환
    - Tree 계열: TreeExplainer → explainer(X)
    - LogisticRegression: LinearExplainer → explainer(X)
    shap_values는 항상 2D (n_samples, n_features) 보장
    """
    mn = str(model_name).strip()

    if mn in ("XGBoost", "LightGBM", "RandomForest"):
        explainer = shap.TreeExplainer(model)
        sv = explainer(X, check_additivity=False)

        # shap.Explanation 내 values 정규화
        vals = sv.values
        if isinstance(vals, list):
            vals = vals[1]
        vals = np.array(vals)
        if vals.ndim == 3:
            vals = vals[:, :, 1]
        if vals.ndim == 1:
            vals = vals.reshape(1, -1)

        # base_values 정규화
        bv = sv.base_values
        if isinstance(bv, np.ndarray) and bv.ndim == 2:
            bv = bv[:, 1]
        elif isinstance(bv, np.ndarray) and bv.ndim == 0:
            bv = np.full(len(X), float(bv))

        return vals, bv

    if mn == "LogisticRegression":
        explainer = shap.LinearExplainer(model, X)
        sv = explainer(X)
        vals = sv.values
        if isinstance(vals, list):
            vals = vals[1]
        vals = np.array(vals)
        if vals.ndim == 3:
            vals = vals[:, :, 1]
        bv = sv.base_values
        if isinstance(bv, np.ndarray) and bv.ndim == 2:
            bv = bv[:, 1]
        return vals, bv

    # fallback: KernelExplainer (느림, 샘플 제한)
    print(f"  [SHAP] {mn} → KernelExplainer (샘플 200개)")
    bg  = shap.sample(X, min(100, len(X)))
    exp = shap.KernelExplainer(model.predict_proba, bg)
    sv  = exp.shap_values(X.iloc[:min(200, len(X))])
    vals = sv[1] if isinstance(sv, list) else sv
    bv   = np.full(len(vals), exp.expected_value[1] if isinstance(exp.expected_value, (list, np.ndarray)) else exp.expected_value)
    return np.array(vals), bv


# ============================================================
# 4. 단일 업종 처리
# ============================================================

def process_one_industry(top10_csv_path):
    info   = parse_industry_info(top10_csv_path)
    prefix = info["prefix"]

    print(f"\n{'='*70}")
    print(f"[업종] {prefix}")
    print(f"{'='*70}")

    # ── Top10 1위 행 로드
    top10 = pd.read_csv(top10_csv_path, index_col=0)
    if len(top10) == 0:
        print(f"  [경고] Top10 파일 비어있음 → skip")
        return None

    row          = top10.iloc[0]
    FEATURE_SET  = row["FeatureSet"]
    FEATURE_FILE = row["FeatureFile"]
    METHOD       = row["Method"]
    SMOTE_RATIO  = row["SMOTE_Ratio"]
    MODEL_NAME   = row["Model"]

    print(f"  Model={MODEL_NAME} | Method={METHOD} | FeatureSet={FEATURE_SET}")

    # ── 파일 존재 확인
    feature_path = os.path.join(info["feature_folder"], FEATURE_FILE)
    score_path   = os.path.join(SCORE_DIR, f"{prefix}_통합_스코어_데이터.csv")
    shap_feat_path = os.path.join(
        SHAP_DIR, prefix, f"feature_analysis_{FEATURE_SET}.csv"
    )

    for label, path in [
        ("Train",        info["train_file"]),
        ("Test",         info["test_file"]),
        ("피처 파일",    feature_path),
        ("스코어 데이터", score_path),
    ]:
        if not os.path.exists(path):
            print(f"  [경고] {label} 없음: {path} → skip")
            return None

    # ── 데이터 로드
    train_full = pd.read_parquet(info["train_file"])
    test       = pd.read_parquet(info["test_file"])
    y_train    = train_full[TARGET_COL]
    y_test     = test[TARGET_COL]
    all_data   = pd.concat([train_full, test], ignore_index=True)

    feat_df      = pd.read_csv(feature_path)
    col_key      = "feature" if "feature" in feat_df.columns else feat_df.columns[0]
    use_features = [f for f in feat_df[col_key].tolist() if f in train_full.columns]
    print(f"  피처 수: {len(use_features)}개 | 전체 데이터: {len(all_data)}행")

    # ── 결측치 처리
    imputer     = SimpleImputer(strategy="median")
    X_train_imp = pd.DataFrame(
        imputer.fit_transform(train_full[use_features]), columns=use_features
    )
    X_test_imp  = pd.DataFrame(
        imputer.transform(test[use_features]), columns=use_features
    )

    # ── 리샘플링 + 모델 학습
    X_res, y_res, pos_weight = apply_resampling(X_train_imp, y_train, METHOD, SMOTE_RATIO)
    model = make_model(MODEL_NAME, pos_weight)
    model.fit(X_res, y_res)
    print(f"  모델 학습 완료 ({MODEL_NAME})")

    # ── Threshold 산출 (Test 기준 Recall >= RECALL_MIN)
    y_prob_test = model.predict_proba(X_test_imp)[:, 1]
    found_thr   = find_threshold_at_recall(y_test, y_prob_test, RECALL_MIN)
    THRESHOLD   = found_thr if found_thr is not None else 0.01
    print(f"  Threshold (Recall>={RECALL_MIN}) = {THRESHOLD:.2f}")

    # ──────────────────────────────────────────────────────────
    # Step A. 통합_스코어_데이터 기반 필터링
    #   2023년 정상(0)인 기업의 2022~2024년 데이터
    # ──────────────────────────────────────────────────────────
    score_df = pd.read_csv(score_path, dtype={ID_COL: str})
    score_df[ID_COL] = score_df[ID_COL].str.zfill(10)

    valid_companies = score_df[
        score_df["연도"] == FILTER_BASE_YEAR
    ].query(f"{TARGET_COL} == 0")[ID_COL].unique()

    filtered_df = score_df[
        score_df[ID_COL].isin(valid_companies) &
        score_df["연도"].isin(FILTER_YEARS)
    ].copy()
    print(f"  필터링: {len(filtered_df)}행 "
          f"(기업 수: {filtered_df[ID_COL].nunique()}개, "
          f"연도: {FILTER_YEARS})")

    # ──────────────────────────────────────────────────────────
    # Step B. 전체 데이터 SHAP 계산 (Waterfall용 개별값)
    # ──────────────────────────────────────────────────────────
    all_feat_imp = pd.DataFrame(
        imputer.transform(all_data[use_features]),
        columns=use_features,
        index=all_data.index
    )

    print(f"  SHAP 계산 중... (전체 {len(all_data)}행)")
    shap_vals, base_vals = get_shap_explanation(model, MODEL_NAME, all_feat_imp)
    print(f"  SHAP 완료. shape={shap_vals.shape}")

    y_prob_all = model.predict_proba(all_feat_imp)[:, 1]
    y_pred_all = (y_prob_all >= THRESHOLD).astype(int)
    feature_arr = np.array(use_features)

    # ── Top N SHAP 추출
    n_rows      = len(all_data)
    top_feature = np.empty((n_rows, N_TOP), dtype=object)
    top_value   = np.full((n_rows, N_TOP), np.nan)
    top_shap    = np.full((n_rows, N_TOP), np.nan)

    for i in range(n_rows):
        order = np.argsort(-np.abs(shap_vals[i]))[:N_TOP]
        k = len(order)
        top_feature[i, :k] = feature_arr[order]
        top_value[i, :k]   = all_feat_imp.iloc[i][feature_arr[order]].values
        top_shap[i, :k]    = shap_vals[i][order]

    # ── shap_summary 생성
    shap_summary = pd.DataFrame({
        ID_COL      : all_data[ID_COL].values if ID_COL in all_data.columns else "",
        COMPANY_COL : all_data[COMPANY_COL].values,
        YEAR_COL    : all_data[YEAR_COL].values,
        "base_value": base_vals,
        "threshold" : THRESHOLD,
        "pred_prob" : y_prob_all,
        "pred_label": y_pred_all,
    })
    for r in range(N_TOP):
        shap_summary[f"Top{r+1}_Feature"]    = top_feature[:, r]
        shap_summary[f"Top{r+1}_Value"]      = top_value[:, r]
        shap_summary[f"Top{r+1}_SHAP_Value"] = top_shap[:, r]

    # 중복 제거 (사업자등록번호 + 회계년도 기준)
    id_col_used = ID_COL if ID_COL in all_data.columns else COMPANY_COL
    shap_summary = shap_summary.drop_duplicates(
        subset=[id_col_used, YEAR_COL], keep="first"
    ).reset_index(drop=True)

    # ──────────────────────────────────────────────────────────
    # Step C. filtered_df + shap_summary 병합 → 기업 파일
    # ──────────────────────────────────────────────────────────
    # filtered_df의 "연도" ↔ shap_summary의 YEAR_COL(회계년도) 매칭
    shap_for_merge = shap_summary.rename(columns={YEAR_COL: "연도"})
    merge_keys = [ID_COL, "연도"] if ID_COL in filtered_df.columns else [COMPANY_COL, "연도"]

    firm_df = filtered_df.merge(
        shap_for_merge,
        on=merge_keys,
        how="left",
        suffixes=("", "_shap")
    )

    # 중복 컬럼 정리
    dup_cols = [c for c in firm_df.columns if c.endswith("_shap")]
    firm_df  = firm_df.drop(columns=dup_cols)

    n_matched   = firm_df["pred_prob"].notna().sum()
    n_unmatched = firm_df["pred_prob"].isna().sum()
    print(f"  SHAP 병합 완료: 매칭 {n_matched}행 / 미매칭 {n_unmatched}행")

    firm_save_path = os.path.join(DASHBOARD_DIR, f"{prefix}_기업.csv")
    firm_df.to_csv(firm_save_path, index=False, encoding="utf-8-sig")
    print(f"  기업 파일 저장 → {firm_save_path}")

    # ──────────────────────────────────────────────────────────
    # Step D. feature_analysis → 핵심피처 추출 → 산업 파일
    # ──────────────────────────────────────────────────────────
    if os.path.exists(shap_feat_path):
        feat_analysis = pd.read_csv(shap_feat_path)
        core_df = feat_analysis[
            feat_analysis["Feature_Type"] == "★ 핵심피처 (SHAP+Perm 모두 높음)"
        ][["Combined_Rank", "Feature", "Category", "Feature_Type"]].reset_index(drop=True)
        print(f"  핵심피처: {len(core_df)}개 추출")
    else:
        print(f"  [경고] feature_analysis 없음: {shap_feat_path} → 빈 산업 파일 생성")
        core_df = pd.DataFrame(
            columns=["Combined_Rank", "Feature", "Category", "Feature_Type"]
        )

    # 업종 정보 추가
    core_df.insert(0, "업종", prefix)
    industry_save_path = os.path.join(DASHBOARD_DIR, f"{prefix}_산업.csv")
    core_df.to_csv(industry_save_path, index=False, encoding="utf-8-sig")
    print(f"  산업 파일 저장 → {industry_save_path}")

    # 메모리 해제
    del train_full, test, all_data, all_feat_imp, shap_vals, model
    gc.collect()

    return {
        "Industry"    : prefix,
        "Model"       : MODEL_NAME,
        "Threshold"   : THRESHOLD,
        "N_Filtered"  : len(filtered_df),
        "N_SHAP_Match": n_matched,
        "N_CoreFeature": len(core_df),
    }


# ============================================================
# 5. 전체 Top10 파일 순회 실행
# ============================================================

def main():
    top10_files = sorted(glob.glob(os.path.join(TOP10_FOLDER, "*.csv")))
    if not top10_files:
        print(f"[오류] Top10 파일이 없습니다: {TOP10_FOLDER}")
        return

    # ── 특정 업종만 실행할 경우 여기에 지정 (빈 리스트면 전체 실행)
    RETRY_ONLY = [
        "M19_도매및소매업",
    ]

    if RETRY_ONLY:
        run_files = [f for f in top10_files
                     if any(kw in os.path.basename(f) for kw in RETRY_ONLY)]
        print(f"[재실행 모드] 대상 업종 수: {len(run_files)}개")
    else:
        run_files = top10_files
        print(f"[전체 실행 모드] 업종 수: {len(run_files)}개")

    for f in run_files:
        print(f"  {os.path.basename(f)}")

    results = []
    failed  = []

    for top10_csv in run_files:
        try:
            res = process_one_industry(top10_csv)
            if res:
                results.append(res)
        except Exception as e:
            name = os.path.basename(top10_csv)
            print(f"\n  [오류] {name} 처리 중 예외: {e}")
            import traceback; traceback.print_exc()
            failed.append({"file": name, "error": str(e)})

    print(f"\n\n{'='*70}")
    print(f"전체 처리 완료: 성공 {len(results)}개 / 실패 {len(failed)}개")
    print(f"{'='*70}")

    if results:
        pd.DataFrame(results).to_csv(
            os.path.join(DASHBOARD_DIR, "전체_업종_대시보드_처리요약.csv"),
            index=False, encoding="utf-8-sig"
        )

    if failed:
        for f in failed:
            print(f"  {f['file']} : {f['error']}")
        pd.DataFrame(failed).to_csv(
            os.path.join(DASHBOARD_DIR, "실패_목록.csv"),
            index=False, encoding="utf-8-sig"
        )


if __name__ == "__main__":
    main()

[재실행 모드] 대상 업종 수: 1개
  M19_도매및소매업_Top10 우수모델.csv

[업종] M19_도매및소매업
  Model=XGBoost | Method=ClassWeight | FeatureSet=top65_dedup52
  피처 수: 52개 | 전체 데이터: 39908행
  모델 학습 완료 (XGBoost)
  Threshold (Recall>=0.9) = 0.34
  필터링: 11271행 (기업 수: 3829개, 연도: [2022, 2023, 2024])
  SHAP 계산 중... (전체 39908행)
  SHAP 완료. shape=(39908, 52)
  SHAP 병합 완료: 매칭 11271행 / 미매칭 0행
  기업 파일 저장 → 중간결과\23_대시보드\M19_도매및소매업_기업.csv
  핵심피처: 9개 추출
  산업 파일 저장 → 중간결과\23_대시보드\M19_도매및소매업_산업.csv


전체 처리 완료: 성공 1개 / 실패 0개
